# 08 — Vedic / Panchang Feature Engineering

## Purpose

This notebook constructs the **F4 Vedic/Panchang feature family** for
the 5-minute SOLETE renewable-energy forecasting dataset.

The aim is not merely to convert Gregorian timestamps into calendar
labels.

Instead, F4 is designed as an additional deterministic temporal
representation that can later be compared against:

- **F0** — historical/autoregressive and meteorological baseline;
- **F1** — Gregorian calendar representation;
- **F2** — independently calculated solar astronomy;
- **F3** — continuous Sun-Moon/lunar astronomy;
- **F4** — Vedic/Panchang representation.

This staged design allows the later forecasting experiments to test
whether Panchang-derived variables provide predictive information
beyond the continuous astronomical quantities from which they are
constructed.


## Astronomical foundation

The astronomical methodology established in
`07_astronomical_feature_engineering.ipynb` is retained.

For the 5-minute SOLETE dataset:

- canonical timeline: **UTC**;
- site: **DTU/SYSLAB, Denmark**;
- latitude: **55.6867° N**;
- longitude: **12.0985° E**;
- astronomical representative time:
  **recorded timestamp + 2 minutes**;
- planetary/lunar ephemeris:
  **JPL DE440s**;
- astronomy library:
  **Skyfield 1.55**.

The supplied SOLETE `Azimuth[deg]` and `Elevation[deg]` variables are
not used as the astronomical basis of the Panchang calculations.


## Sidereal reference convention

Several Panchang quantities require a clearly specified sidereal
(`nirayana`) reference.

The primary sidereal convention adopted in this study is the
**standard Lahiri ayanamsha**.

For numerical reproducibility, the study will use the standard Lahiri
definition implemented by Swiss Ephemeris as:

`SIDM_LAHIRI`

The ayanamsha convention is therefore treated as an explicit
methodological parameter rather than being inherited invisibly from
a calendar or Panchang software package.

Although Lahiri is often associated with the Citra/Chitrapaksha
tradition, this notebook does **not** treat standard Lahiri as identical
to the separate astronomical convention sometimes called
**True Chitrapaksha**.

The exact implementation and numerical values of the selected
ayanamsha will be validated before any ayanamsha-dependent Panchang
feature is constructed.


## Tropical and sidereal longitude

The astronomical Sun and Moon positions obtained from Skyfield are
initially expressed using the validated apparent geocentric ecliptic
framework.

For Panchang quantities requiring sidereal longitude, the general
relationship will be:

`sidereal longitude = tropical longitude - ayanamsha`

with the resulting longitude wrapped into the interval:

`[0°, 360°)`

The precise numerical ayanamsha will be obtained from the explicitly
selected standard Lahiri implementation.


## Dependence of Panchang quantities on ayanamsha

Not every Panchang element depends on the sidereal reference in the
same way.


### Tithi

Tithi is based on the Moon-Sun angular separation:

`(Moon longitude - Sun longitude) mod 360°`

The full 360-degree synodic cycle is divided into 30 equal intervals,
each spanning:

`12°`

Because the same common ayanamsha shift would be subtracted from both
solar and lunar longitude, it cancels in their difference.

Therefore:

**Tithi is independent of the choice of a common ayanamsha.**


### Paksha

Paksha divides the lunar cycle into two halves:

- **Shukla Paksha** — waxing half;
- **Krishna Paksha** — waning half.

Paksha is derived directly from Tithi / Moon-Sun angular separation.

Therefore:

**Paksha is also independent of the choice of a common ayanamsha.**


### Karana

A Karana corresponds to half of a Tithi.

Each half-Tithi spans:

`6°`

of Moon-Sun angular separation.

Therefore:

**Karana is also independent of the choice of a common ayanamsha.**


### Nakshatra

Nakshatra divides the **sidereal lunar longitude** into 27 equal
segments.

Each Nakshatra spans:

`360° / 27 = 13°20′`

Therefore:

**Nakshatra depends on the adopted ayanamsha.**


### Yoga

Yoga is based on the wrapped sum of the **sidereal solar longitude**
and **sidereal lunar longitude**.

The resulting 360-degree circle is divided into 27 equal segments.

Therefore:

**Yoga depends on the adopted ayanamsha.**


### Masa

Lunar month (`Masa`) requires additional calendrical conventions beyond
the instantaneous Sun-Moon geometry.

These include choices such as:

- new-moon or full-moon month boundaries;
- **amanta** versus **purnimanta** month convention;
- handling of unusual cases such as repeated or omitted solar-sign
  transitions.

Masa will therefore be handled separately only after the core Panchang
quantities have been implemented and validated.


## Initial F4 scope

The first validated F4 implementation will focus on:

1. **Tithi**
2. **Paksha**
3. **Nakshatra**
4. **Yoga**
5. **Karana**

Masa and other higher-level calendrical quantities will not be added
until their conventions have been explicitly defined and validated.


## Relationship between F3 and F4

A major methodological objective of this project is to avoid giving F4
astronomical information that was absent from the preceding control.

The relevant relationship is:

### F3 continuous astronomy

F3 already contains continuous quantities describing:

- Moon-Sun phase angle;
- lunar ecliptic longitude;
- solar ecliptic longitude;
- lunar latitude;
- illuminated fraction;
- Earth-Moon distance.


### F4 Panchang representation

F4 transforms related astronomical geometry into structured calendrical
representations such as:

- Tithi;
- Paksha;
- Nakshatra;
- Yoga;
- Karana.

This design allows the later experiment:

`F3 → F4`

to test whether the Panchang representation adds forecasting value
beyond the underlying continuous astronomical quantities.


## Research principle

No opaque Panchang package will be treated as the primary source of the
research features.

Where practical, Panchang quantities will be derived explicitly from
the already validated astronomical Sun-Moon geometry.

External astronomical or Panchang software may be used as an
**independent validation reference**, but the mathematical construction
of the model features will remain documented and reproducible.

In [1]:
# ============================================================
# CELL 1 — LOAD SOLETE DATA AND ESTABLISH F4 ASSUMPTIONS
# ============================================================

# ------------------------------------------------------------
# PURPOSE
# ------------------------------------------------------------
#
# Notebook 08 begins from the methodological results already
# established in Notebooks 06 and 07.
#
# This cell:
#
#     1. identifies the project root;
#     2. makes the project source package importable;
#     3. loads the same 5-minute SOLETE dataset;
#     4. verifies its timeline;
#     5. recreates the final astronomical reference time:
#
#            SOLETE timestamp + 2 minutes
#
# No Panchang quantity is calculated yet.


from pathlib import Path
import sys

import numpy as np
import pandas as pd


# ============================================================
# 1. IDENTIFY PROJECT ROOT
# ============================================================
#
# When this notebook is executed from:
#
#     Project/notebooks/
#
# Python's default module search path does not necessarily
# include:
#
#     Project/
#
# We therefore determine the project root explicitly and add
# it to sys.path before importing reusable project modules.

PROJECT_ROOT = Path.cwd().resolve()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent


if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(
        0,
        str(PROJECT_ROOT)
    )


print("Project root:")
print(PROJECT_ROOT)


# ============================================================
# 2. IMPORT REUSABLE PROJECT LOADER
# ============================================================

from src.data_loader import load_solete


print("\nProject data loader imported successfully:")
print(load_solete is not None)


# ============================================================
# 2. LOCATE THE 5-MINUTE SOLETE HDF5 FILE
# ============================================================
#
# We search inside the project dataset directory rather than
# duplicating an absolute machine-specific path.
#
# Exactly one 5-minute HDF5 file should be found.

dataset_root = (
    PROJECT_ROOT
    / "solete_dataset"
)


five_minute_candidates = sorted(
    list(
        dataset_root.glob(
            "**/*5min*.h5"
        )
    )
    +
    list(
        dataset_root.glob(
            "**/*5_min*.h5"
        )
    )
)


print("\n5-minute HDF5 candidates:")
print(five_minute_candidates)


assert len(five_minute_candidates) == 1, (
    "Expected exactly one 5-minute SOLETE HDF5 file, "
    f"but found {len(five_minute_candidates)}."
)


DATA_FILE = five_minute_candidates[0]


print("\nSelected dataset file:")
print(DATA_FILE)


# ============================================================
# 3. LOAD THE 5-MINUTE DATASET
# ============================================================

df_raw = load_solete(
    DATA_FILE
)


print("\nDataset shape:")
print(
    df_raw.shape
)


print("\nTimestamp range:")
print(
    df_raw.index.min(),
    "to",
    df_raw.index.max()
)


# ============================================================
# 4. VERIFY THE EXPECTED 5-MINUTE TIMELINE
# ============================================================

print("\nTimezone:")
print(
    df_raw.index.tz
)


print("\nChronological:")
print(
    df_raw.index.is_monotonic_increasing
)


print("\nDuplicate timestamps:")
print(
    df_raw.index.duplicated().sum()
)


expected_timeline = pd.date_range(
    start=df_raw.index.min(),
    end=df_raw.index.max(),
    freq="5min",
    tz="UTC",
)


missing_timestamps = (
    expected_timeline
    .difference(
        df_raw.index
    )
)


print("\nMissing 5-minute timestamps:")
print(
    len(missing_timestamps)
)


# ============================================================
# 5. RECREATE FINAL ASTRONOMICAL REFERENCE TIME
# ============================================================
#
# Notebook 07 established that the 5-minute SOLETE values
# closely follow averages of samples at:
#
#     t, t+1, t+2, t+3, t+4 minutes
#
# Therefore, the discrete temporal centre is:
#
#     t + 2 minutes
#
# F4 uses exactly the same astronomical reference convention
# as F2 and F3.

ASTRONOMICAL_OFFSET_MINUTES = 2


astro_reference_times = (
    df_raw.index
    + pd.Timedelta(
        minutes=ASTRONOMICAL_OFFSET_MINUTES
    )
)


print("\nAstronomical reference convention:")
print(
    f"SOLETE timestamp + "
    f"{ASTRONOMICAL_OFFSET_MINUTES} minutes"
)


print("\nFirst dataset timestamp:")
print(
    df_raw.index[0]
)


print("\nFirst astronomical reference time:")
print(
    astro_reference_times[0]
)

Project root:
/Users/siddharthgupta/Downloads/NSUT/Semester 1/MTITC102 AI and Machine Learning/Project

Project data loader imported successfully:
True

5-minute HDF5 candidates:
[PosixPath('/Users/siddharthgupta/Downloads/NSUT/Semester 1/MTITC102 AI and Machine Learning/Project/solete_dataset/SOLETE_Pombo_5min.h5')]

Selected dataset file:
/Users/siddharthgupta/Downloads/NSUT/Semester 1/MTITC102 AI and Machine Learning/Project/solete_dataset/SOLETE_Pombo_5min.h5

Dataset shape:
(131617, 11)

Timestamp range:
2018-06-01 00:00:00+00:00 to 2019-09-01 00:00:00+00:00

Timezone:
UTC

Chronological:
True

Duplicate timestamps:
0

Missing 5-minute timestamps:
0

Astronomical reference convention:
SOLETE timestamp + 2 minutes

First dataset timestamp:
2018-06-01 00:00:00+00:00

First astronomical reference time:
2018-06-01 00:02:00+00:00


## Re-establishing the Astronomical Engine

F4 is derived from the same independently validated Sun-Moon astronomy
used for F3.

The JPL DE440s ephemeris and Skyfield time system are therefore loaded
again in this notebook.

The astronomical foundation remains:

- JPL DE440s;
- Skyfield;
- geocentric Sun-Moon geometry;
- UTC timeline;
- `t + 2 minutes` astronomical representative time.

This section does **not** yet construct sidereal longitude.

The selected sidereal convention is recorded as:

**standard Lahiri ayanamsha**

with the intended numerical reference implementation:

**Swiss Ephemeris `SIDM_LAHIRI`**

Skyfield continues to provide the underlying Sun-Moon astronomical
geometry. The Lahiri ayanamsha will be introduced explicitly and
validated separately before Nakshatra or Yoga are calculated.

In [2]:
# ============================================================
# CELL 2 — INITIALIZE SKYFIELD, DE440s, AND F4 CONVENTIONS
# ============================================================

# ------------------------------------------------------------
# PURPOSE
# ------------------------------------------------------------
#
# F4 must use exactly the same astronomical foundation that
# was validated for F3 in Notebook 07.
#
# This cell therefore:
#
#     1. records the Skyfield version;
#     2. locates the project-local JPL DE440s ephemeris;
#     3. creates the Skyfield time scale;
#     4. loads Earth, Moon, and Sun;
#     5. converts the complete t+2-minute reference timeline
#        into Skyfield time objects;
#     6. records the selected sidereal convention.
#
# IMPORTANT:
#
# No ayanamsha value is calculated here.
#
# No Tithi, Paksha, Nakshatra, Yoga, Karana, or Masa is
# calculated here.
#
# The Lahiri ayanamsha implementation will be introduced and
# validated explicitly in a later cell.


import skyfield

from skyfield.api import Loader


# ============================================================
# 1. RECORD SKYFIELD VERSION
# ============================================================
#
# Software versions are recorded because astronomical
# calculations should remain reproducible.

print("Skyfield version:")
print(
    skyfield.__version__
)


# ============================================================
# 2. LOCATE PROJECT-LOCAL JPL EPHEMERIS
# ============================================================
#
# Notebook 07 downloaded:
#
#     de440s.bsp
#
# into:
#
#     Project/ephemeris/
#
# The ephemeris directory itself is excluded from Git because
# the BSP file is external astronomical reference data.

EPHEMERIS_DIR = (
    PROJECT_ROOT
    / "ephemeris"
)


EPHEMERIS_FILE = (
    EPHEMERIS_DIR
    / "de440s.bsp"
)


print("\nEphemeris file:")
print(
    EPHEMERIS_FILE
)


print("\nEphemeris exists:")
print(
    EPHEMERIS_FILE.exists()
)


assert EPHEMERIS_FILE.exists(), (
    "DE440s ephemeris was not found in the project "
    "ephemeris directory."
)


# ============================================================
# 3. CREATE PROJECT-LOCAL SKYFIELD LOADER
# ============================================================
#
# Using a project-local Loader ensures that Skyfield reuses
# the same DE440s file used in Notebook 07.

skyfield_loader = Loader(
    str(EPHEMERIS_DIR)
)


# ============================================================
# 4. CREATE SKYFIELD TIME SCALE
# ============================================================

ts = skyfield_loader.timescale()


print("\nSkyfield time scale created:")
print(
    ts is not None
)


# ============================================================
# 5. LOAD JPL DE440s
# ============================================================

eph = skyfield_loader(
    "de440s.bsp"
)


# ------------------------------------------------------------
# Retrieve only the celestial bodies required for the
# Panchang astronomical foundation.
# ------------------------------------------------------------

earth = eph["Earth"]
moon = eph["Moon"]
sun = eph["Sun"]


print("\nRequired bodies loaded:")

print(
    "Earth:",
    earth is not None
)

print(
    "Moon:",
    moon is not None
)

print(
    "Sun:",
    sun is not None
)


# ============================================================
# 6. CONVERT FULL ASTRONOMICAL TIMELINE TO SKYFIELD
# ============================================================
#
# Notebook 07 established the representative time:
#
#     SOLETE timestamp + 2 minutes
#
# astro_reference_times was recreated in Cell 1.
#
# The pandas timestamps are timezone-aware UTC values.
# Conversion to Python datetime objects preserves those
# physical UTC instants for Skyfield.

skyfield_times = ts.from_datetimes(
    astro_reference_times.to_pydatetime()
)


print("\nNumber of Skyfield timestamps:")
print(
    len(astro_reference_times)
)


print("\nFirst Skyfield UTC timestamp:")
print(
    skyfield_times[0].utc_iso()
)


print("\nLast Skyfield UTC timestamp:")
print(
    skyfield_times[-1].utc_iso()
)


# ============================================================
# 7. RECORD SIDEREAL / AYANAMSHA CONVENTION
# ============================================================
#
# Some Panchang quantities require sidereal longitude.
#
# The primary convention selected for this research is:
#
#     Standard Lahiri ayanamsha
#
# For numerical reproducibility, the intended reference
# implementation is:
#
#     Swiss Ephemeris SIDM_LAHIRI
#
# We deliberately avoid the ambiguous shorthand
# "Lahiri / Chitrapaksha" because multiple related
# Chitrapaksha/Lahiri definitions exist.
#
# The numerical ayanamsha itself will be calculated and
# validated in a later cell.

AYANAMSHA_NAME = (
    "Standard Lahiri"
)


AYANAMSHA_REFERENCE_IMPLEMENTATION = (
    "Swiss Ephemeris SIDM_LAHIRI"
)


print("\nSelected sidereal convention:")
print(
    AYANAMSHA_NAME
)


print("\nReference implementation:")
print(
    AYANAMSHA_REFERENCE_IMPLEMENTATION
)


# ============================================================
# 8. RECORD WHICH CORE PANCHANG ELEMENTS REQUIRE AYANAMSHA
# ============================================================
#
# This is methodological metadata only.
#
# Tithi:
#     Moon - Sun longitude difference
#     -> common ayanamsha cancels
#
# Paksha:
#     derived from Tithi
#     -> ayanamsha independent
#
# Karana:
#     half-Tithi
#     -> ayanamsha independent
#
# Nakshatra:
#     sidereal lunar longitude
#     -> ayanamsha dependent
#
# Yoga:
#     sidereal Sun + sidereal Moon longitude
#     -> ayanamsha dependent

PANCHANG_AYANAMSHA_DEPENDENCE = {
    "Tithi": False,
    "Paksha": False,
    "Karana": False,
    "Nakshatra": True,
    "Yoga": True,
}


print("\nCore Panchang ayanamsha dependence:")

for element, depends_on_ayanamsha in (
    PANCHANG_AYANAMSHA_DEPENDENCE.items()
):

    print(
        f"{element:10s}: "
        f"{depends_on_ayanamsha}"
    )

Skyfield version:
1.55

Ephemeris file:
/Users/siddharthgupta/Downloads/NSUT/Semester 1/MTITC102 AI and Machine Learning/Project/ephemeris/de440s.bsp

Ephemeris exists:
True

Skyfield time scale created:
True

Required bodies loaded:
Earth: True
Moon: True
Sun: True

Number of Skyfield timestamps:
131617

First Skyfield UTC timestamp:
2018-06-01T00:02:00Z

Last Skyfield UTC timestamp:
2019-09-01T00:02:00Z

Selected sidereal convention:
Standard Lahiri

Reference implementation:
Swiss Ephemeris SIDM_LAHIRI

Core Panchang ayanamsha dependence:
Tithi     : False
Paksha    : False
Karana    : False
Nakshatra : True
Yoga      : True


## Lahiri Ayanamsha Sanity Check

Before constructing any sidereal Panchang quantity, the selected
standard Lahiri ayanamsha is evaluated on a small set of representative
timestamps.

Swiss Ephemeris `SIDM_LAHIRI` is used as the numerical reference
convention.

Two ayanamsha values are retained during validation:

1. `get_ayanamsa_ut()` — the older/simple value without nutation;
2. `get_ayanamsa_ex_ut()` — the extended value, including nutation
   because `FLG_NONUT` is not requested.

Cross-engine validation later in this notebook showed that the extended
value reproduces direct sidereal calculations substantially more closely
near Panchang sector boundaries.

Therefore, the **extended Lahiri ayanamsha is the canonical value used
for the final F4 Nakshatra and Yoga construction**.

The simple value is retained only as a diagnostic reference.

Skyfield + JPL DE440s remains the common source of Sun-Moon
astronomical geometry for F3 and F4.

In [3]:
# ============================================================
# CELL 3 — VERIFY CANONICAL EXTENDED LAHIRI AYANAMSHA
# ============================================================

# ------------------------------------------------------------
# PURPOSE
# ------------------------------------------------------------
#
# This cell establishes the canonical Lahiri ayanamsha
# convention used by the final F4 implementation.
#
# Swiss Ephemeris mode:
#
#     SIDM_LAHIRI
#
# We calculate:
#
#     1. simple / legacy ayanamsha
#        -> get_ayanamsa_ut()
#
#     2. extended ayanamsha
#        -> get_ayanamsa_ex_ut()
#
# The EXTENDED value is canonical for Nakshatra and Yoga.
#
# No Panchang feature is calculated in this cell.


import swisseph as swe


# ============================================================
# 1. RECORD SWISS EPHEMERIS VERSION
# ============================================================

print("Swiss Ephemeris version:")
print(
    swe.version
)


# ============================================================
# 2. SELECT STANDARD LAHIRI SIDEREAL MODE
# ============================================================

swe.set_sid_mode(
    swe.SIDM_LAHIRI
)


print("\nSwiss Ephemeris Lahiri mode selected:")
print(
    swe.SIDM_LAHIRI
)


# ============================================================
# 3. DEFINE REPRESENTATIVE TEST UTC TIMES
# ============================================================
#
# These span the SOLETE observation period.
#
# They use the same dataset-specific astronomical
# representative-time convention:
#
#     SOLETE timestamp + 2 minutes

ayanamsha_test_times = pd.DatetimeIndex(
    [
        "2018-06-01 00:02:00+00:00",
        "2018-09-01 12:02:00+00:00",
        "2018-12-01 00:02:00+00:00",
        "2019-03-01 12:02:00+00:00",
        "2019-06-01 00:02:00+00:00",
        "2019-09-01 00:02:00+00:00",
    ]
)


# ============================================================
# 4. CONVERT TEST TIMESTAMPS TO JULIAN DAY
# ============================================================
#
# This retains the same Julian-Day construction used during
# the earlier notebook validation so that the sidereal-method
# correction is isolated from any unrelated change.

ayanamsha_test_jd = []


for timestamp in ayanamsha_test_times:

    decimal_hour = (
        timestamp.hour
        + timestamp.minute / 60
        + timestamp.second / 3600
        + timestamp.microsecond / 3_600_000_000
    )

    jd_ut = swe.julday(
        timestamp.year,
        timestamp.month,
        timestamp.day,
        decimal_hour,
        swe.GREG_CAL,
    )

    ayanamsha_test_jd.append(
        jd_ut
    )


ayanamsha_test_jd = np.array(
    ayanamsha_test_jd,
    dtype=float,
)


# ============================================================
# 5. CALCULATE SIMPLE / LEGACY LAHIRI AYANAMSHA
# ============================================================
#
# This older function returns ayanamsha without nutation.
#
# It is retained only for audit comparison.

lahiri_ayanamsha_simple_test_deg = np.array(
    [
        swe.get_ayanamsa_ut(
            jd_ut
        )
        for jd_ut in ayanamsha_test_jd
    ],
    dtype=float,
)


# ============================================================
# 6. CALCULATE EXTENDED LAHIRI AYANAMSHA
# ============================================================
#
# get_ayanamsa_ex_ut() accepts calculation flags.
#
# Because FLG_NONUT is NOT supplied, the returned ayanamsha
# includes nutation.
#
# This is the CANONICAL Lahiri value used by the final F4
# sidereal transformation.

lahiri_ayanamsha_test_deg = np.empty(
    len(
        ayanamsha_test_jd
    ),
    dtype=float,
)


lahiri_ayanamsha_test_retflags = np.empty(
    len(
        ayanamsha_test_jd
    ),
    dtype=int,
)


for i, jd_ut in enumerate(
    ayanamsha_test_jd
):

    retflag, ayanamsha_extended = (
        swe.get_ayanamsa_ex_ut(
            jd_ut,
            swe.FLG_SWIEPH,
        )
    )

    lahiri_ayanamsha_test_retflags[i] = (
        retflag
    )

    lahiri_ayanamsha_test_deg[i] = (
        ayanamsha_extended
    )


# ============================================================
# 7. COMPARE SIMPLE AND CANONICAL EXTENDED VALUES
# ============================================================

ayanamsha_test_difference_deg = (
    lahiri_ayanamsha_test_deg
    -
    lahiri_ayanamsha_simple_test_deg
)


ayanamsha_test_difference_arcsec = (
    ayanamsha_test_difference_deg
    * 3600.0
)


# ============================================================
# 8. BASIC VALIDATION
# ============================================================

print(
    "\nAll canonical extended ayanamsha values finite:"
)

print(
    np.isfinite(
        lahiri_ayanamsha_test_deg
    ).all()
)


print(
    "\nCanonical extended Lahiri range [deg]:"
)

print(
    lahiri_ayanamsha_test_deg.min(),
    "to",
    lahiri_ayanamsha_test_deg.max(),
)


print(
    "\nCanonical Lahiri change across test period [deg]:"
)

print(
    lahiri_ayanamsha_test_deg[-1]
    -
    lahiri_ayanamsha_test_deg[0]
)


print(
    "\nMaximum absolute extended-vs-simple difference [deg]:"
)

print(
    np.abs(
        ayanamsha_test_difference_deg
    ).max()
)


print(
    "\nMaximum absolute extended-vs-simple difference [arcsec]:"
)

print(
    np.abs(
        ayanamsha_test_difference_arcsec
    ).max()
)


print(
    "\nExtended ayanamsha return flags:"
)

print(
    np.unique(
        lahiri_ayanamsha_test_retflags
    )
)


# ============================================================
# 9. CREATE INSPECTION TABLE
# ============================================================

ayanamsha_test_results = pd.DataFrame(
    {
        "UTC_timestamp":
            ayanamsha_test_times,

        "julian_day_ut":
            ayanamsha_test_jd,

        "lahiri_simple_deg":
            lahiri_ayanamsha_simple_test_deg,

        "lahiri_extended_canonical_deg":
            lahiri_ayanamsha_test_deg,

        "extended_minus_simple_arcsec":
            ayanamsha_test_difference_arcsec,
    }
)


print(
    "\nCanonical Lahiri ayanamsha sanity check:"
)

ayanamsha_test_results

Swiss Ephemeris version:
2.10.03

Swiss Ephemeris Lahiri mode selected:
1

All canonical extended ayanamsha values finite:
True

Canonical extended Lahiri range [deg]:
24.110343268621794 to 24.127372016642582

Canonical Lahiri change across test period [deg]:
0.017028748020788242

Maximum absolute extended-vs-simple difference [deg]:
0.004820112217142736

Maximum absolute extended-vs-simple difference [arcsec]:
17.35240398171385

Extended ayanamsha return flags:
[2]

Canonical Lahiri ayanamsha sanity check:


,UTC_timestamp,julian_day_ut,lahiri_simple_deg,lahiri_extended_canonical_deg,extended_minus_simple_arcsec
0,2018-06-01 00:02:00+00:00,2.458271e+06,24.114318,24.110343,-14.307597
1,2018-09-01 12:02:00+00:00,2.458363e+06,24.117856,24.113967,-13.999092
2,2018-12-01 00:02:00+00:00,2.458454e+06,24.121317,24.116863,-16.033889
3,2019-03-01 12:02:00+00:00,2.458544e+06,24.124778,24.120681,-14.751250
4,2019-06-01 00:02:00+00:00,2.458636e+06,24.128278,24.123458,-17.352404
5,2019-09-01 00:02:00+00:00,2.458728e+06,24.131797,24.127372,-15.929565


## Constructing Lahiri Sidereal Longitudes

The canonical standard Lahiri ayanamsha has now been established using
Swiss Ephemeris `SIDM_LAHIRI` with the extended
`get_ayanamsa_ex_ut()` calculation.

Skyfield + JPL DE440s supplies the apparent geocentric tropical
ecliptic longitudes of the Sun and Moon.

For F4, Lahiri-aligned sidereal longitude is constructed as:

`sidereal longitude = tropical longitude - extended Lahiri ayanamsha`

wrapped into `[0°, 360°)`.

This cell validates two properties on the six representative timestamps:

1. the tropical-to-sidereal angular shift reproduces the canonical
   extended Lahiri ayanamsha;

2. subtracting the same ayanamsha from both Sun and Moon leaves their
   angular separation unchanged.

The second property explains why Tithi, Paksha and Karana are
ayanamsha-independent even though Nakshatra and Yoga are not.

In [4]:
# ============================================================
# CELL 4 — CONSTRUCT AND VALIDATE CANONICAL SIDEREAL LONGITUDES
# ============================================================

# ------------------------------------------------------------
# PURPOSE
# ------------------------------------------------------------
#
# We now apply the canonical EXTENDED Lahiri ayanamsha from
# Cell 3 to Skyfield's apparent geocentric tropical ecliptic
# longitudes.
#
# Final F4 sidereal convention:
#
#     sidereal longitude
#         =
#     Skyfield tropical longitude
#         -
#     extended Lahiri ayanamsha
#
# wrapped into:
#
#     [0°, 360°)
#
#
# We validate:
#
#     1. the applied tropical -> sidereal shift equals the
#        canonical Lahiri ayanamsha;
#
#     2. Moon-Sun angular separation is unchanged.
#
# The second property demonstrates why:
#
#     Tithi
#     Paksha
#     Karana
#
# are independent of a common ayanamsha.


from skyfield.framelib import ecliptic_frame


# ============================================================
# 1. CONVERT THE SIX TEST TIMESTAMPS TO SKYFIELD TIMES
# ============================================================

skyfield_ayanamsha_test_times = (
    ts.from_datetimes(
        ayanamsha_test_times.to_pydatetime()
    )
)


# ============================================================
# 2. CALCULATE APPARENT GEOCENTRIC SUN AND MOON POSITIONS
# ============================================================

earth_at_ayanamsha_test = earth.at(
    skyfield_ayanamsha_test_times
)


moon_apparent_ayanamsha_test = (
    earth_at_ayanamsha_test
    .observe(moon)
    .apparent()
)


sun_apparent_ayanamsha_test = (
    earth_at_ayanamsha_test
    .observe(sun)
    .apparent()
)


# ============================================================
# 3. EXTRACT TROPICAL ECLIPTIC LONGITUDES
# ============================================================

(
    moon_ecliptic_lat_test,
    moon_ecliptic_lon_test,
    moon_distance_test,
) = moon_apparent_ayanamsha_test.frame_latlon(
    ecliptic_frame
)


(
    sun_ecliptic_lat_test,
    sun_ecliptic_lon_test,
    sun_distance_test,
) = sun_apparent_ayanamsha_test.frame_latlon(
    ecliptic_frame
)


moon_tropical_longitude_deg = (
    moon_ecliptic_lon_test.degrees
    % 360.0
)


sun_tropical_longitude_deg = (
    sun_ecliptic_lon_test.degrees
    % 360.0
)


# ============================================================
# 4. APPLY CANONICAL EXTENDED LAHIRI AYANAMSHA
# ============================================================
#
# lahiri_ayanamsha_test_deg now refers to the canonical
# EXTENDED Lahiri value established in Cell 3.

moon_sidereal_longitude_deg = (
    (
        moon_tropical_longitude_deg
        -
        lahiri_ayanamsha_test_deg
    )
    % 360.0
)


sun_sidereal_longitude_deg = (
    (
        sun_tropical_longitude_deg
        -
        lahiri_ayanamsha_test_deg
    )
    % 360.0
)


# ============================================================
# 5. VERIFY THE APPLIED TROPICAL -> SIDEREAL SHIFT
# ============================================================
#
# Circularly:
#
#     tropical longitude - sidereal longitude
#
# should reproduce the canonical Lahiri ayanamsha.

moon_applied_shift_deg = (
    (
        moon_tropical_longitude_deg
        -
        moon_sidereal_longitude_deg
    )
    % 360.0
)


sun_applied_shift_deg = (
    (
        sun_tropical_longitude_deg
        -
        sun_sidereal_longitude_deg
    )
    % 360.0
)


moon_shift_error_deg = np.abs(
    moon_applied_shift_deg
    -
    lahiri_ayanamsha_test_deg
)


sun_shift_error_deg = np.abs(
    sun_applied_shift_deg
    -
    lahiri_ayanamsha_test_deg
)


print(
    "Maximum Moon tropical-to-sidereal "
    "shift error [deg]:"
)

print(
    moon_shift_error_deg.max()
)


print(
    "\nMaximum Sun tropical-to-sidereal "
    "shift error [deg]:"
)

print(
    sun_shift_error_deg.max()
)


# ============================================================
# 6. CALCULATE MOON-SUN SEPARATION
# ============================================================

phase_tropical_deg = (
    (
        moon_tropical_longitude_deg
        -
        sun_tropical_longitude_deg
    )
    % 360.0
)


phase_sidereal_deg = (
    (
        moon_sidereal_longitude_deg
        -
        sun_sidereal_longitude_deg
    )
    % 360.0
)


# ============================================================
# 7. VERIFY PHASE INVARIANCE
# ============================================================
#
# Use the smallest circular angular difference.

phase_invariance_error_deg = np.abs(
    (
        phase_sidereal_deg
        -
        phase_tropical_deg
        +
        180.0
    )
    % 360.0
    -
    180.0
)


print(
    "\nMaximum Moon-Sun phase invariance error [deg]:"
)

print(
    phase_invariance_error_deg.max()
)


# ============================================================
# 8. BUILD VALIDATION TABLE
# ============================================================

sidereal_longitude_test_results = pd.DataFrame(
    {
        "UTC_timestamp":
            ayanamsha_test_times,

        "lahiri_extended_deg":
            lahiri_ayanamsha_test_deg,

        "moon_tropical_lon_deg":
            moon_tropical_longitude_deg,

        "moon_sidereal_lon_deg":
            moon_sidereal_longitude_deg,

        "sun_tropical_lon_deg":
            sun_tropical_longitude_deg,

        "sun_sidereal_lon_deg":
            sun_sidereal_longitude_deg,

        "moon_sun_phase_tropical_deg":
            phase_tropical_deg,

        "moon_sun_phase_sidereal_deg":
            phase_sidereal_deg,
    }
)


# ============================================================
# 9. DISPLAY RESULTS
# ============================================================

print(
    "\nCanonical Lahiri sidereal-longitude validation:"
)

sidereal_longitude_test_results

Maximum Moon tropical-to-sidereal shift error [deg]:
1.4210854715202004e-14

Maximum Sun tropical-to-sidereal shift error [deg]:
1.4210854715202004e-14

Maximum Moon-Sun phase invariance error [deg]:
2.842170943040401e-14

Canonical Lahiri sidereal-longitude validation:


,UTC_timestamp,lahiri_extended_deg,moon_tropical_lon_deg,moon_sidereal_lon_deg,sun_tropical_lon_deg,sun_sidereal_lon_deg,moon_sun_phase_tropical_deg,moon_sun_phase_sidereal_deg
0,2018-06-01 00:02:00+00:00,24.110343,277.274831,253.164488,70.472685,46.362341,206.802147,206.802147
1,2018-09-01 12:02:00+00:00,24.113967,48.839032,24.725065,159.004618,134.890651,249.834414,249.834414
2,2018-12-01 00:02:00+00:00,24.116863,171.499213,147.382350,248.725946,224.609083,282.773267,282.773267
3,2019-03-01 12:02:00+00:00,24.120681,284.655756,260.535075,340.605945,316.485264,304.049810,304.049810
4,2019-06-01 00:02:00+00:00,24.123458,40.332366,16.208908,70.245502,46.122044,330.086865,330.086865
5,2019-09-01 00:02:00+00:00,24.127372,180.567801,156.440429,158.288139,134.160767,22.279661,22.279661


## Tithi Construction

Tithi is derived from the angular separation between the Moon and Sun.

The continuous Moon-Sun phase angle is:

`phase = (Moon longitude - Sun longitude) mod 360°`

The full 360-degree synodic cycle is divided into 30 equal Tithis.

Therefore, each Tithi spans:

`360° / 30 = 12°`

The numerical Tithi index is:

`floor(phase / 12°) + 1`

giving values from:

`1` to `30`

The first 15 Tithis correspond to the waxing half of the lunar cycle
(**Shukla Paksha**), while Tithis 16–30 correspond to the waning half
(**Krishna Paksha**).

This calculation is independent of the selected ayanamsha because it
uses only the Moon-Sun longitude difference.

In [5]:
# ============================================================
# CELL 5 — CALCULATE AND INSPECT TITHI
# ============================================================

# ------------------------------------------------------------
# PURPOSE
# ------------------------------------------------------------
#
# This is the first actual Panchang quantity constructed in
# Notebook 08.
#
# Tithi is based only on the Moon-Sun angular separation:
#
#     phase = (Moon longitude - Sun longitude) mod 360
#
# The 360-degree synodic cycle is divided into:
#
#     30 Tithis
#
# Each Tithi spans:
#
#     12 degrees
#
# Tithi numbering:
#
#      1–15  -> Shukla Paksha
#     16–30  -> Krishna Paksha
#
# No ayanamsha is required.


# ============================================================
# 1. DEFINE TITHI WIDTH
# ============================================================

TITHI_WIDTH_DEG = 12.0


print("Tithi angular width [deg]:")
print(
    TITHI_WIDTH_DEG
)


# ============================================================
# 2. CALCULATE TITHI NUMBER
# ============================================================
#
# phase_tropical_deg was already verified in Cell 4.
#
# Formula:
#
#     floor(phase / 12) + 1
#
# Expected range:
#
#     1 ... 30

tithi_number_test = (
    np.floor(
        phase_tropical_deg
        / TITHI_WIDTH_DEG
    )
    .astype(int)
    + 1
)


# ============================================================
# 3. CALCULATE POSITION WITHIN CURRENT TITHI
# ============================================================
#
# This continuous value is useful for validation.
#
# Range:
#
#     0 <= position < 12 degrees

tithi_position_deg_test = (
    phase_tropical_deg
    % TITHI_WIDTH_DEG
)


# ------------------------------------------------------------
# Fractional progress through current Tithi:
#
#     0 -> beginning
#     1 -> approaching next Tithi

tithi_progress_test = (
    tithi_position_deg_test
    / TITHI_WIDTH_DEG
)


# ============================================================
# 4. DERIVE PAKSHA
# ============================================================
#
# Tithi 1–15:
#     Shukla Paksha
#
# Tithi 16–30:
#     Krishna Paksha

paksha_test = np.where(
    tithi_number_test <= 15,
    "Shukla",
    "Krishna",
)


# ============================================================
# 5. NUMBER TITHI WITHIN EACH PAKSHA
# ============================================================
#
# This gives:
#
#     1 ... 15
#
# within either Shukla or Krishna Paksha.

tithi_in_paksha_test = (
    (
        tithi_number_test
        - 1
    )
    % 15
    + 1
)


# ============================================================
# 6. BASIC VALIDATION
# ============================================================

print("\nTithi number range:")
print(
    tithi_number_test.min(),
    "to",
    tithi_number_test.max(),
)


print("\nAll Tithi numbers inside 1–30:")
print(
    (
        (tithi_number_test >= 1)
        &
        (tithi_number_test <= 30)
    ).all()
)


print("\nAll Tithi progress values inside [0, 1):")
print(
    (
        (tithi_progress_test >= 0)
        &
        (tithi_progress_test < 1)
    ).all()
)


# ============================================================
# 7. CREATE TEST TABLE
# ============================================================

tithi_test_results = pd.DataFrame(
    {
        "UTC_timestamp":
            ayanamsha_test_times,

        "moon_sun_phase_deg":
            phase_tropical_deg,

        "tithi_number":
            tithi_number_test,

        "paksha":
            paksha_test,

        "tithi_in_paksha":
            tithi_in_paksha_test,

        "tithi_position_deg":
            tithi_position_deg_test,

        "tithi_progress":
            tithi_progress_test,
    }
)


# ============================================================
# 8. DISPLAY RESULTS
# ============================================================

print("\nTithi sanity check:")

tithi_test_results

Tithi angular width [deg]:
12.0

Tithi number range:
2 to 28

All Tithi numbers inside 1–30:
True

All Tithi progress values inside [0, 1):
True

Tithi sanity check:


,UTC_timestamp,moon_sun_phase_deg,tithi_number,paksha,tithi_in_paksha,tithi_position_deg,tithi_progress
0,2018-06-01 00:02:00+00:00,206.802147,18,Krishna,3,2.802147,0.233512
1,2018-09-01 12:02:00+00:00,249.834414,21,Krishna,6,9.834414,0.819535
2,2018-12-01 00:02:00+00:00,282.773267,24,Krishna,9,6.773267,0.564439
3,2019-03-01 12:02:00+00:00,304.049810,26,Krishna,11,4.049810,0.337484
4,2019-06-01 00:02:00+00:00,330.086865,28,Krishna,13,6.086865,0.507239
5,2019-09-01 00:02:00+00:00,22.279661,2,Shukla,2,10.279661,0.856638


### Tithi Naming and Boundary Validation

The numerical Tithi calculation must also behave correctly at the
boundaries between lunar phases.

The adopted interval convention is:

- Tithi 1 begins at `0°`;
- each Tithi occupies a left-inclusive 12-degree interval;
- at exactly `180°`, Shukla Purnima has ended and Krishna Pratipada
  begins;
- at `360°`, the phase wraps to `0°`, beginning the next Shukla
  Pratipada.

Traditional Tithi names are retained as human-readable audit labels.

For Tithis 1–14 within either Paksha, the names repeat. The fifteenth
Tithi is:

- **Purnima** in Shukla Paksha;
- **Amavasya** in Krishna Paksha.

These names are diagnostic labels at this stage and are not yet the
final model encoding.

In [6]:
# ============================================================
# CELL 6 — TITHI NAMES AND BOUNDARY VALIDATION
# ============================================================

# ------------------------------------------------------------
# PURPOSE
# ------------------------------------------------------------
#
# Cell 5 established the numerical Tithi calculation.
#
# We now:
#
#     1. map within-Paksha Tithi numbers to traditional
#        human-readable names;
#     2. verify important 12-degree boundaries explicitly;
#     3. confirm correct behaviour around Full Moon and
#        New Moon transitions.
#
# These names are audit / interpretation labels.
#
# No model encoding is selected yet.


# ============================================================
# 1. DEFINE TITHI NAMES FOR POSITIONS 1–14
# ============================================================

TITHI_BASE_NAMES = {
    1:  "Pratipada",
    2:  "Dvitiya",
    3:  "Tritiya",
    4:  "Chaturthi",
    5:  "Panchami",
    6:  "Shashthi",
    7:  "Saptami",
    8:  "Ashtami",
    9:  "Navami",
    10: "Dashami",
    11: "Ekadashi",
    12: "Dvadashi",
    13: "Trayodashi",
    14: "Chaturdashi",
}


# ============================================================
# 2. CREATE A SMALL TITHI-NAME FUNCTION
# ============================================================

def get_tithi_name(
    tithi_in_paksha,
    paksha,
):
    """
    Return a human-readable Tithi name.

    Parameters
    ----------
    tithi_in_paksha : int
        Position from 1 to 15 within the current Paksha.

    paksha : str
        Either "Shukla" or "Krishna".

    Returns
    -------
    str
        Traditional Tithi name.
    """

    if tithi_in_paksha == 15:

        if paksha == "Shukla":
            return "Purnima"

        if paksha == "Krishna":
            return "Amavasya"

    return TITHI_BASE_NAMES[
        int(tithi_in_paksha)
    ]


# ============================================================
# 3. ADD NAMES TO THE SIX REAL TEST TIMESTAMPS
# ============================================================

tithi_name_test = [
    get_tithi_name(
        tithi_in_paksha,
        paksha,
    )
    for tithi_in_paksha, paksha
    in zip(
        tithi_in_paksha_test,
        paksha_test,
    )
]


tithi_test_results[
    "tithi_name"
] = tithi_name_test


print(
    "Named Tithi sanity check:\n"
)

print(
    tithi_test_results[
        [
            "UTC_timestamp",
            "moon_sun_phase_deg",
            "tithi_number",
            "paksha",
            "tithi_in_paksha",
            "tithi_name",
        ]
    ]
)


# ============================================================
# 4. DEFINE ARTIFICIAL PHASE-ANGLE BOUNDARY TESTS
# ============================================================
#
# We deliberately test values immediately before and exactly
# at important 12-degree boundaries.
#
# Examples:
#
#     0°       -> Tithi 1
#     12°      -> Tithi 2
#
#     just below 180°
#              -> Tithi 15 / Purnima
#
#     exactly 180°
#              -> Tithi 16 / Krishna Pratipada
#
#     just below 360°
#              -> Tithi 30 / Amavasya
#
#     360°
#              -> wraps to 0° / Tithi 1

boundary_phase_input_deg = np.array(
    [
        0.0,
        11.999999,
        12.0,
        167.999999,
        168.0,
        179.999999,
        180.0,
        191.999999,
        192.0,
        347.999999,
        348.0,
        359.999999,
        360.0,
    ]
)


# ------------------------------------------------------------
# Wrap every phase into [0°, 360°).
# ------------------------------------------------------------

boundary_phase_deg = (
    boundary_phase_input_deg
    % 360
)


# ============================================================
# 5. CALCULATE BOUNDARY TITHIS
# ============================================================

boundary_tithi_number = (
    np.floor(
        boundary_phase_deg
        / TITHI_WIDTH_DEG
    )
    .astype(int)
    + 1
)


boundary_paksha = np.where(
    boundary_tithi_number <= 15,
    "Shukla",
    "Krishna",
)


boundary_tithi_in_paksha = (
    (
        boundary_tithi_number
        - 1
    )
    % 15
    + 1
)


boundary_tithi_name = [
    get_tithi_name(
        tithi_in_paksha,
        paksha,
    )
    for tithi_in_paksha, paksha
    in zip(
        boundary_tithi_in_paksha,
        boundary_paksha,
    )
]


# ============================================================
# 6. BUILD BOUNDARY VALIDATION TABLE
# ============================================================

tithi_boundary_results = pd.DataFrame(
    {
        "input_phase_deg":
            boundary_phase_input_deg,

        "wrapped_phase_deg":
            boundary_phase_deg,

        "tithi_number":
            boundary_tithi_number,

        "paksha":
            boundary_paksha,

        "tithi_in_paksha":
            boundary_tithi_in_paksha,

        "tithi_name":
            boundary_tithi_name,
    }
)


# ============================================================
# 7. VALIDATE FULL TITHI RANGE IN THE BOUNDARY TEST
# ============================================================

print(
    "\nBoundary-test Tithi range:"
)

print(
    boundary_tithi_number.min(),
    "to",
    boundary_tithi_number.max(),
)


print(
    "\nTithi boundary validation:"
)

tithi_boundary_results

Named Tithi sanity check:

              UTC_timestamp  moon_sun_phase_deg  tithi_number   paksha  \
0 2018-06-01 00:02:00+00:00          206.802147            18  Krishna   
1 2018-09-01 12:02:00+00:00          249.834414            21  Krishna   
2 2018-12-01 00:02:00+00:00          282.773267            24  Krishna   
3 2019-03-01 12:02:00+00:00          304.049810            26  Krishna   
4 2019-06-01 00:02:00+00:00          330.086865            28  Krishna   
5 2019-09-01 00:02:00+00:00           22.279661             2   Shukla   

   tithi_in_paksha  tithi_name  
0                3     Tritiya  
1                6    Shashthi  
2                9      Navami  
3               11    Ekadashi  
4               13  Trayodashi  
5                2     Dvitiya  

Boundary-test Tithi range:
1 to 30

Tithi boundary validation:


,input_phase_deg,wrapped_phase_deg,tithi_number,paksha,tithi_in_paksha,tithi_name
0,0.000000,0.000000,1,Shukla,1,Pratipada
1,11.999999,11.999999,1,Shukla,1,Pratipada
2,12.000000,12.000000,2,Shukla,2,Dvitiya
3,167.999999,167.999999,14,Shukla,14,Chaturdashi
4,168.000000,168.000000,15,Shukla,15,Purnima
5,179.999999,179.999999,15,Shukla,15,Purnima
6,180.000000,180.000000,16,Krishna,1,Pratipada
7,191.999999,191.999999,16,Krishna,1,Pratipada
8,192.000000,192.000000,17,Krishna,2,Dvitiya
9,347.999999,347.999999,29,Krishna,14,Chaturdashi


## Full-Timeline Tithi and Paksha Generation

The Tithi calculation has now been validated on both real timestamps and
explicit angular boundary cases.

It is therefore applied to the complete 5-minute SOLETE timeline.

For every timestamp:

1. apparent geocentric lunar and solar ecliptic longitudes are calculated
   using Skyfield + JPL DE440s;
2. Moon-Sun angular separation is wrapped into `[0°, 360°)`;
3. the phase is divided into thirty 12-degree Tithi intervals;
4. Paksha and the within-Paksha Tithi number are derived;
5. traditional Tithi names are retained as audit labels.

The continuous Moon-Sun phase and within-Tithi progress are retained for
validation and reproducibility.

They are **not automatically treated as additional F4 model inputs**,
because F3 already contains the underlying continuous lunar-phase
geometry.

This preserves a controlled comparison between:

**F3 — continuous astronomical representation**

and

**F4 — structured Panchang representation**.

In [7]:
# ============================================================
# CELL 7 — GENERATE FULL TITHI AND PAKSHA TIMELINE
# ============================================================

# ------------------------------------------------------------
# PURPOSE
# ------------------------------------------------------------
#
# The Tithi/Paksha formulation has now been validated.
#
# We therefore calculate it for every one of the 131,617
# SOLETE 5-minute timestamps.
#
# Astronomical basis:
#
#     Skyfield
#     JPL DE440s
#     apparent geocentric ecliptic longitude
#     astronomical reference time = t + 2 minutes
#
# Tithi itself does NOT require the Lahiri ayanamsha.
#
# We create:
#
#     df_tithi
#
# containing both audit quantities and Panchang labels.
#
# No Nakshatra, Yoga, or Karana is calculated in this cell.


from skyfield.framelib import ecliptic_frame


# ============================================================
# 1. CALCULATE FULL APPARENT GEOCENTRIC POSITIONS
# ============================================================

earth_at_full_times = earth.at(
    skyfield_times
)


moon_apparent_full = (
    earth_at_full_times
    .observe(moon)
    .apparent()
)


sun_apparent_full = (
    earth_at_full_times
    .observe(sun)
    .apparent()
)


# ============================================================
# 2. EXTRACT FULL TROPICAL ECLIPTIC LONGITUDES
# ============================================================

(
    moon_ecliptic_lat_full,
    moon_ecliptic_lon_full,
    moon_distance_full,
) = moon_apparent_full.frame_latlon(
    ecliptic_frame
)


(
    sun_ecliptic_lat_full,
    sun_ecliptic_lon_full,
    sun_distance_full,
) = sun_apparent_full.frame_latlon(
    ecliptic_frame
)


moon_tropical_longitude_full_deg = (
    moon_ecliptic_lon_full.degrees
    % 360
)


sun_tropical_longitude_full_deg = (
    sun_ecliptic_lon_full.degrees
    % 360
)


# ============================================================
# 3. CALCULATE FULL MOON-SUN PHASE
# ============================================================
#
# This is the same continuous phase quantity validated in
# Notebook 07 and again in this notebook.

moon_sun_phase_full_deg = (
    (
        moon_tropical_longitude_full_deg
        -
        sun_tropical_longitude_full_deg
    )
    % 360
)


# ============================================================
# 4. CALCULATE TITHI NUMBER
# ============================================================

tithi_number_full = (
    np.floor(
        moon_sun_phase_full_deg
        / TITHI_WIDTH_DEG
    )
    .astype(int)
    + 1
)


# ============================================================
# 5. DERIVE PAKSHA
# ============================================================

paksha_full = np.where(
    tithi_number_full <= 15,
    "Shukla",
    "Krishna",
)


# ============================================================
# 6. CALCULATE TITHI NUMBER WITHIN PAKSHA
# ============================================================

tithi_in_paksha_full = (
    (
        tithi_number_full
        - 1
    )
    % 15
    + 1
)


# ============================================================
# 7. ADD TRADITIONAL TITHI NAMES
# ============================================================

tithi_name_full = [
    get_tithi_name(
        tithi_in_paksha,
        paksha,
    )
    for tithi_in_paksha, paksha
    in zip(
        tithi_in_paksha_full,
        paksha_full,
    )
]


# ============================================================
# 8. CALCULATE WITHIN-TITHI POSITION
# ============================================================
#
# These continuous quantities are retained primarily for
# validation / audit purposes.
#
# They are not automatically part of the F4 modelling set.

tithi_position_full_deg = (
    moon_sun_phase_full_deg
    % TITHI_WIDTH_DEG
)


tithi_progress_full = (
    tithi_position_full_deg
    / TITHI_WIDTH_DEG
)


# ============================================================
# 9. BUILD FULL TITHI / PAKSHA TABLE
# ============================================================

df_tithi = pd.DataFrame(
    {
        # ---------------------------------------------
        # Audit astronomical quantity
        # ---------------------------------------------
        "moon_sun_phase_deg":
            moon_sun_phase_full_deg,

        # ---------------------------------------------
        # Panchang quantities
        # ---------------------------------------------
        "tithi_number":
            tithi_number_full,

        "paksha":
            paksha_full,

        "tithi_in_paksha":
            tithi_in_paksha_full,

        "tithi_name":
            tithi_name_full,

        # ---------------------------------------------
        # Audit / diagnostic quantities
        # ---------------------------------------------
        "tithi_position_deg":
            tithi_position_full_deg,

        "tithi_progress":
            tithi_progress_full,
    },
    index=df_raw.index,
)


# ============================================================
# 10. BASIC VALIDATION
# ============================================================

print("Tithi table shape:")
print(
    df_tithi.shape
)


print("\nMissing values:")
print(
    df_tithi.isna().sum()
)


print("\nTithi number range:")
print(
    df_tithi["tithi_number"].min(),
    "to",
    df_tithi["tithi_number"].max(),
)


print("\nUnique Tithi numbers:")
print(
    df_tithi["tithi_number"].nunique()
)


print("\nUnique within-Paksha Tithi numbers:")
print(
    df_tithi["tithi_in_paksha"].nunique()
)


print("\nPaksha values:")
print(
    df_tithi["paksha"].value_counts()
)


print("\nTithi-progress range:")
print(
    df_tithi["tithi_progress"].min(),
    "to",
    df_tithi["tithi_progress"].max(),
)


# ============================================================
# 11. CHECK ALL 30 TITHIS ARE PRESENT
# ============================================================

expected_tithis = set(
    range(1, 31)
)


observed_tithis = set(
    df_tithi[
        "tithi_number"
    ].unique()
)


missing_tithis = sorted(
    expected_tithis
    -
    observed_tithis
)


print("\nMissing Tithi numbers from full timeline:")
print(
    missing_tithis
)


# ============================================================
# 12. SHOW OBSERVATION COUNTS BY TITHI
# ============================================================
#
# Counts will not be perfectly equal because:
#
#     - Tithi duration varies slightly;
#     - the dataset starts/ends partway through lunar cycles;
#     - timestamps are sampled every five minutes.

tithi_counts = (
    df_tithi[
        "tithi_number"
    ]
    .value_counts()
    .sort_index()
)


print("\nObservation counts by Tithi:")
print(
    tithi_counts
)


# ============================================================
# 13. DISPLAY SAMPLE
# ============================================================

print("\nSample full Tithi/Paksha table:")

df_tithi.head(12)

Tithi table shape:
(131617, 7)

Missing values:
moon_sun_phase_deg    0
tithi_number          0
paksha                0
tithi_in_paksha       0
tithi_name            0
tithi_position_deg    0
tithi_progress        0
dtype: int64

Tithi number range:
1 to 30

Unique Tithi numbers:
30

Unique within-Paksha Tithi numbers:
15

Paksha values:
paksha
Krishna    67545
Shukla     64072
Name: count, dtype: int64

Tithi-progress range:
2.3497901994555073e-06 to 0.9999780633871183

Missing Tithi numbers from full timeline:
[]

Observation counts by Tithi:
tithi_number
1     4386
2     4361
3     4179
4     4206
5     4237
6     4271
7     4294
8     4308
9     4309
10    4302
11    4278
12    4258
13    4241
14    4223
15    4219
16    4217
17    4236
18    4498
19    4596
20    4627
21    4637
22    4649
23    4637
24    4612
25    4577
26    4534
27    4482
28    4444
29    4412
30    4387
Name: count, dtype: int64

Sample full Tithi/Paksha table:


,moon_sun_phase_deg,tithi_number,paksha,tithi_in_paksha,tithi_name,tithi_position_deg,tithi_progress
Timestamp,,,,,,,
2018-06-01 00:00:00+00:00,206.802147,18,Krishna,3,Tritiya,2.802147,0.233512
2018-06-01 00:05:00+00:00,206.840226,18,Krishna,3,Tritiya,2.840226,0.236685
2018-06-01 00:10:00+00:00,206.878304,18,Krishna,3,Tritiya,2.878304,0.239859
2018-06-01 00:15:00+00:00,206.916380,18,Krishna,3,Tritiya,2.916380,0.243032
2018-06-01 00:20:00+00:00,206.954455,18,Krishna,3,Tritiya,2.954455,0.246205
2018-06-01 00:25:00+00:00,206.992529,18,Krishna,3,Tritiya,2.992529,0.249377
2018-06-01 00:30:00+00:00,207.030601,18,Krishna,3,Tritiya,3.030601,0.252550
2018-06-01 00:35:00+00:00,207.068671,18,Krishna,3,Tritiya,3.068671,0.255723
2018-06-01 00:40:00+00:00,207.106740,18,Krishna,3,Tritiya,3.106740,0.258895


## Karana Construction

A Karana corresponds to half of a Tithi.

Since each Tithi spans `12°` of Moon-Sun angular separation, each
Karana spans:

`6°`

The complete synodic cycle therefore contains:

`360° / 6° = 60`

sequential Karana intervals.

Eleven traditional Karana names are used.

### Fixed and movable Karanas

The first half of Shukla Pratipada is:

- **Kimstughna**

The following seven movable Karanas repeat eight times:

1. **Bava**
2. **Balava**
3. **Kaulava**
4. **Taitila**
5. **Gara**
6. **Vanija**
7. **Vishti**

The final three half-Tithi intervals are:

- **Shakuni**
- **Chatushpada**
- **Naga**

The sequence then wraps back to Kimstughna at the beginning of the next
synodic cycle.

Karana depends only on Moon-Sun angular separation and therefore does
not require an ayanamsha.

In [8]:
# ============================================================
# CELL 8 — VALIDATE KARANA SEQUENCE
# ============================================================

# ------------------------------------------------------------
# PURPOSE
# ------------------------------------------------------------
#
# Karana is based on half-Tithi intervals.
#
# Each Karana spans:
#
#     6 degrees
#
# of Moon-Sun angular separation.
#
# Therefore the complete synodic cycle contains:
#
#     60 Karana slots
#
# Before applying Karana to all 131,617 timestamps, we first
# verify the traditional fixed/movable sequence explicitly.
#
# No full-timeline Karana calculation is performed yet.


# ============================================================
# 1. DEFINE KARANA ANGULAR WIDTH
# ============================================================

KARANA_WIDTH_DEG = 6.0


print("Karana angular width [deg]:")
print(
    KARANA_WIDTH_DEG
)


# ============================================================
# 2. DEFINE MOVABLE KARANA CYCLE
# ============================================================

MOVABLE_KARANA_NAMES = [
    "Bava",
    "Balava",
    "Kaulava",
    "Taitila",
    "Gara",
    "Vanija",
    "Vishti",
]


# ============================================================
# 3. DEFINE KARANA-NAME FUNCTION
# ============================================================
#
# Karana slot numbering:
#
#      1     -> Kimstughna
#
#      2–57  -> seven movable Karanas repeated eight times
#
#      58    -> Shakuni
#      59    -> Chatushpada
#      60    -> Naga

def get_karana_name(
    karana_slot,
):
    """
    Return the traditional Karana name for a slot from 1 to 60.
    """

    karana_slot = int(
        karana_slot
    )

    if karana_slot == 1:
        return "Kimstughna"

    if 2 <= karana_slot <= 57:

        movable_index = (
            karana_slot - 2
        ) % 7

        return MOVABLE_KARANA_NAMES[
            movable_index
        ]

    if karana_slot == 58:
        return "Shakuni"

    if karana_slot == 59:
        return "Chatushpada"

    if karana_slot == 60:
        return "Naga"

    raise ValueError(
        "Karana slot must be between 1 and 60."
    )


# ============================================================
# 4. CONSTRUCT ALL 60 KARANA SLOTS
# ============================================================

karana_slots_test = np.arange(
    1,
    61,
)


karana_names_test = [
    get_karana_name(
        slot
    )
    for slot in karana_slots_test
]


# ============================================================
# 5. ASSOCIATE EACH SLOT WITH ITS PHASE INTERVAL
# ============================================================
#
# Slot 1:
#
#     [0°, 6°)
#
# Slot 2:
#
#     [6°, 12°)
#
# ...
#
# Slot 60:
#
#     [354°, 360°)

karana_start_deg = (
    (
        karana_slots_test
        - 1
    )
    * KARANA_WIDTH_DEG
)


karana_end_deg = (
    karana_slots_test
    * KARANA_WIDTH_DEG
)


# ============================================================
# 6. BUILD VALIDATION TABLE
# ============================================================

karana_sequence_test = pd.DataFrame(
    {
        "karana_slot":
            karana_slots_test,

        "phase_start_deg":
            karana_start_deg,

        "phase_end_deg":
            karana_end_deg,

        "karana_name":
            karana_names_test,
    }
)


# ============================================================
# 7. VERIFY NAME COUNTS
# ============================================================

print("\nKarana name counts:")
print(
    karana_sequence_test[
        "karana_name"
    ].value_counts()
)


print("\nNumber of unique Karana names:")
print(
    karana_sequence_test[
        "karana_name"
    ].nunique()
)


# ============================================================
# 8. DISPLAY BEGINNING AND END OF SEQUENCE
# ============================================================

print("\nFirst 12 Karana slots:")
print(
    karana_sequence_test.head(12)
)


print("\nLast 12 Karana slots:")

karana_sequence_test.tail(12)

Karana angular width [deg]:
6.0

Karana name counts:
karana_name
Bava           8
Balava         8
Kaulava        8
Taitila        8
Gara           8
Vanija         8
Vishti         8
Kimstughna     1
Shakuni        1
Chatushpada    1
Naga           1
Name: count, dtype: int64

Number of unique Karana names:
11

First 12 Karana slots:
    karana_slot  phase_start_deg  phase_end_deg karana_name
0             1              0.0            6.0  Kimstughna
1             2              6.0           12.0        Bava
2             3             12.0           18.0      Balava
3             4             18.0           24.0     Kaulava
4             5             24.0           30.0     Taitila
5             6             30.0           36.0        Gara
6             7             36.0           42.0      Vanija
7             8             42.0           48.0      Vishti
8             9             48.0           54.0        Bava
9            10             54.0           60.0      Balava
10 

,karana_slot,phase_start_deg,phase_end_deg,karana_name
48,49,288.0,294.0,Vanija
49,50,294.0,300.0,Vishti
50,51,300.0,306.0,Bava
51,52,306.0,312.0,Balava
52,53,312.0,318.0,Kaulava
53,54,318.0,324.0,Taitila
54,55,324.0,330.0,Gara
55,56,330.0,336.0,Vanija
56,57,336.0,342.0,Vishti
57,58,342.0,348.0,Shakuni


## Full-Timeline Karana Generation

The traditional 60-slot Karana sequence has been validated explicitly.

Karana is now calculated for every SOLETE timestamp directly from the
already validated Moon-Sun angular separation.

For a phase angle in `[0°, 360°)`:

`Karana slot = floor(phase / 6°) + 1`

giving a slot number from `1` to `60`.

The corresponding traditional Karana name is then assigned using the
validated fixed/movable sequence.

The half of the current Tithi is also retained:

- `1` — first half of the Tithi;
- `2` — second half of the Tithi.

Continuous within-Karana position and progress are retained for audit
and validation only. They are not automatically included as F4 model
features because F3 already contains the underlying continuous
Moon-Sun phase information.

In [9]:
# ============================================================
# CELL 9 — GENERATE FULL KARANA TIMELINE
# ============================================================

# ------------------------------------------------------------
# PURPOSE
# ------------------------------------------------------------
#
# Cell 8 validated the complete traditional 60-slot Karana
# sequence.
#
# We now apply that sequence to every 5-minute SOLETE row.
#
# Karana depends only on:
#
#     Moon-Sun angular separation
#
# Each Karana spans:
#
#     6 degrees
#
# Therefore:
#
#     karana_slot = floor(phase / 6) + 1
#
# Expected range:
#
#     1 ... 60
#
# No ayanamsha is required.


# ============================================================
# 1. CALCULATE KARANA SLOT
# ============================================================

karana_slot_full = (
    np.floor(
        moon_sun_phase_full_deg
        / KARANA_WIDTH_DEG
    )
    .astype(int)
    + 1
)


# ============================================================
# 2. ASSIGN TRADITIONAL KARANA NAME
# ============================================================

karana_name_full = [
    get_karana_name(
        slot
    )
    for slot in karana_slot_full
]


# ============================================================
# 3. DETERMINE HALF OF CURRENT TITHI
# ============================================================
#
# Each 12-degree Tithi contains two 6-degree Karanas.
#
# Therefore:
#
#     odd Karana slot  -> first half of a Tithi
#     even Karana slot -> second half of a Tithi
#
# This pattern remains valid across the complete 60-slot
# synodic cycle.

karana_half_in_tithi_full = (
    (
        karana_slot_full
        - 1
    )
    % 2
    + 1
)


# ============================================================
# 4. CALCULATE WITHIN-KARANA POSITION
# ============================================================
#
# Audit / validation quantities only.
#
# Position:
#
#     0 <= value < 6 degrees
#
# Progress:
#
#     0 <= value < 1

karana_position_full_deg = (
    moon_sun_phase_full_deg
    % KARANA_WIDTH_DEG
)


karana_progress_full = (
    karana_position_full_deg
    / KARANA_WIDTH_DEG
)


# ============================================================
# 5. BUILD FULL KARANA TABLE
# ============================================================

df_karana = pd.DataFrame(
    {
        "karana_slot":
            karana_slot_full,

        "karana_name":
            karana_name_full,

        "karana_half_in_tithi":
            karana_half_in_tithi_full,

        # ---------------------------------------------
        # Audit / diagnostic quantities
        # ---------------------------------------------
        "karana_position_deg":
            karana_position_full_deg,

        "karana_progress":
            karana_progress_full,
    },
    index=df_raw.index,
)


# ============================================================
# 6. BASIC VALIDATION
# ============================================================

print("Karana table shape:")
print(
    df_karana.shape
)


print("\nMissing values:")
print(
    df_karana.isna().sum()
)


print("\nKarana slot range:")
print(
    df_karana["karana_slot"].min(),
    "to",
    df_karana["karana_slot"].max(),
)


print("\nUnique Karana slots:")
print(
    df_karana["karana_slot"].nunique()
)


print("\nUnique Karana names:")
print(
    df_karana["karana_name"].nunique()
)


print("\nKarana half-in-Tithi values:")
print(
    df_karana[
        "karana_half_in_tithi"
    ].value_counts().sort_index()
)


print("\nKarana-progress range:")
print(
    df_karana["karana_progress"].min(),
    "to",
    df_karana["karana_progress"].max(),
)


# ============================================================
# 7. CHECK ALL 60 SLOTS ARE PRESENT
# ============================================================

expected_karana_slots = set(
    range(1, 61)
)


observed_karana_slots = set(
    df_karana[
        "karana_slot"
    ].unique()
)


missing_karana_slots = sorted(
    expected_karana_slots
    -
    observed_karana_slots
)


print("\nMissing Karana slots:")
print(
    missing_karana_slots
)


# ============================================================
# 8. OBSERVATION COUNTS BY KARANA NAME
# ============================================================
#
# These are counts of 5-minute SOLETE observations, not the
# number of traditional sequence appearances.
#
# They therefore need not have the 8:1 ratio seen in the
# abstract 60-slot sequence from Cell 8.

karana_name_counts = (
    df_karana[
        "karana_name"
    ]
    .value_counts()
)


print("\nObservation counts by Karana name:")
print(
    karana_name_counts
)


# ============================================================
# 9. CROSS-CHECK KARANA AGAINST TITHI
# ============================================================
#
# Since every Tithi contains exactly two Karana slots:
#
#     Tithi number
#         = floor((Karana slot - 1) / 2) + 1
#
# This provides an independent consistency check.

tithi_from_karana_slot = (
    np.floor(
        (
            karana_slot_full
            - 1
        )
        / 2
    )
    .astype(int)
    + 1
)


karana_tithi_mismatch_count = (
    tithi_from_karana_slot
    != tithi_number_full
).sum()


print(
    "\nKarana-to-Tithi mismatch count:"
)

print(
    karana_tithi_mismatch_count
)


# ============================================================
# 10. DISPLAY SAMPLE
# ============================================================

print("\nSample full Karana table:")

df_karana.head(12)

Karana table shape:
(131617, 5)

Missing values:
karana_slot             0
karana_name             0
karana_half_in_tithi    0
karana_position_deg     0
karana_progress         0
dtype: int64

Karana slot range:
1 to 60

Unique Karana slots:
60

Unique Karana names:
11

Karana half-in-Tithi values:
karana_half_in_tithi
1    65814
2    65803
Name: count, dtype: int64

Karana-progress range:
2.1820016418890495e-06 to 0.9999608621079688

Missing Karana slots:
[]

Observation counts by Karana name:
karana_name
Vishti         17614
Bava           17591
Balava         17590
Kaulava        17564
Vanija         17540
Taitila        17469
Gara           17467
Shakuni         2203
Chatushpada     2196
Kimstughna      2192
Naga            2191
Name: count, dtype: int64

Karana-to-Tithi mismatch count:
0

Sample full Karana table:


,karana_slot,karana_name,karana_half_in_tithi,karana_position_deg,karana_progress
Timestamp,,,,,
2018-06-01 00:00:00+00:00,35,Vanija,1,2.802147,0.467024
2018-06-01 00:05:00+00:00,35,Vanija,1,2.840226,0.473371
2018-06-01 00:10:00+00:00,35,Vanija,1,2.878304,0.479717
2018-06-01 00:15:00+00:00,35,Vanija,1,2.916380,0.486063
2018-06-01 00:20:00+00:00,35,Vanija,1,2.954455,0.492409
2018-06-01 00:25:00+00:00,35,Vanija,1,2.992529,0.498755
2018-06-01 00:30:00+00:00,35,Vanija,1,3.030601,0.505100
2018-06-01 00:35:00+00:00,35,Vanija,1,3.068671,0.511445
2018-06-01 00:40:00+00:00,35,Vanija,1,3.106740,0.517790


## Nakshatra Construction and Boundary Validation

Nakshatra is the first Panchang quantity in this notebook that depends
on the adopted sidereal reference.

The Moon's apparent geocentric tropical ecliptic longitude is supplied
by Skyfield + JPL DE440s.

It is converted to the project's canonical Lahiri-aligned sidereal
longitude using:

`sidereal Moon longitude = tropical Moon longitude - extended Lahiri ayanamsha`

where the extended Lahiri ayanamsha is obtained from Swiss Ephemeris
`SIDM_LAHIRI` using `get_ayanamsa_ex_ut()`.

The sidereal zodiac is divided into 27 equal Nakshatras:

`360° / 27 = 13°20′`

The numerical Nakshatra index is:

`floor(sidereal lunar longitude / (360° / 27)) + 1`

giving values from `1` to `27`.

Nakshatra is therefore explicitly ayanamsha-dependent.

Before applying this definition to the complete SOLETE timeline, the
27-name sequence and angular boundary behavior are validated using the
six representative timestamps established earlier.

In [10]:
# ============================================================
# CELL 10 — CANONICAL NAKSHATRA SANITY AND BOUNDARY VALIDATION
# ============================================================

# ------------------------------------------------------------
# PURPOSE
# ------------------------------------------------------------
#
# Nakshatra depends on the Moon's absolute SIDEREAL
# longitude.
#
# This cell uses the canonical sidereal lunar longitude
# established in Cell 4:
#
#     Skyfield apparent tropical lunar longitude
#         -
#     extended Lahiri ayanamsha
#
# We validate:
#
#     1. the 27-name Nakshatra sequence;
#     2. assignments at six real timestamps;
#     3. the [0°, 360°) angular boundaries.
#
# No full-timeline Nakshatra calculation is performed here.


# ============================================================
# 1. DEFINE NAKSHATRA ANGULAR WIDTH
# ============================================================

NAKSHATRA_COUNT = 27


NAKSHATRA_WIDTH_DEG = (
    360.0
    / NAKSHATRA_COUNT
)


print("Number of Nakshatras:")
print(
    NAKSHATRA_COUNT
)


print("\nNakshatra angular width [deg]:")
print(
    NAKSHATRA_WIDTH_DEG
)


print("\nNakshatra angular width [deg + arcmin]:")
print(
    "13°20′"
)


# ============================================================
# 2. DEFINE THE 27 NAKSHATRA NAMES
# ============================================================

NAKSHATRA_NAMES = [
    "Ashwini",
    "Bharani",
    "Krittika",
    "Rohini",
    "Mrigashirsha",
    "Ardra",
    "Punarvasu",
    "Pushya",
    "Ashlesha",
    "Magha",
    "Purva Phalguni",
    "Uttara Phalguni",
    "Hasta",
    "Chitra",
    "Swati",
    "Vishakha",
    "Anuradha",
    "Jyeshtha",
    "Mula",
    "Purva Ashadha",
    "Uttara Ashadha",
    "Shravana",
    "Dhanishta",
    "Shatabhisha",
    "Purva Bhadrapada",
    "Uttara Bhadrapada",
    "Revati",
]


assert len(
    NAKSHATRA_NAMES
) == NAKSHATRA_COUNT


# ============================================================
# 3. CALCULATE NAKSHATRA FOR SIX REAL TEST TIMESTAMPS
# ============================================================
#
# moon_sidereal_longitude_deg now contains the canonical
# extended-Lahiri sidereal lunar longitude from Cell 4.

nakshatra_number_test = (
    np.floor(
        moon_sidereal_longitude_deg
        / NAKSHATRA_WIDTH_DEG
    )
    .astype(int)
    + 1
)


nakshatra_name_test = [
    NAKSHATRA_NAMES[
        number - 1
    ]
    for number in nakshatra_number_test
]


# ============================================================
# 4. CALCULATE POSITION WITHIN CURRENT NAKSHATRA
# ============================================================

nakshatra_position_deg_test = (
    moon_sidereal_longitude_deg
    % NAKSHATRA_WIDTH_DEG
)


nakshatra_progress_test = (
    nakshatra_position_deg_test
    / NAKSHATRA_WIDTH_DEG
)


# ============================================================
# 5. BASIC VALIDATION
# ============================================================

print(
    "\nNakshatra number range in real test timestamps:"
)

print(
    nakshatra_number_test.min(),
    "to",
    nakshatra_number_test.max(),
)


print(
    "\nAll Nakshatra numbers inside 1–27:"
)

print(
    (
        (nakshatra_number_test >= 1)
        &
        (nakshatra_number_test <= 27)
    ).all()
)


print(
    "\nAll Nakshatra progress values inside [0, 1):"
)

print(
    (
        (nakshatra_progress_test >= 0)
        &
        (nakshatra_progress_test < 1)
    ).all()
)


# ============================================================
# 6. BUILD REAL-TIMESTAMP VALIDATION TABLE
# ============================================================

nakshatra_test_results = pd.DataFrame(
    {
        "UTC_timestamp":
            ayanamsha_test_times,

        "lahiri_extended_deg":
            lahiri_ayanamsha_test_deg,

        "moon_tropical_lon_deg":
            moon_tropical_longitude_deg,

        "moon_sidereal_lon_deg":
            moon_sidereal_longitude_deg,

        "nakshatra_number":
            nakshatra_number_test,

        "nakshatra_name":
            nakshatra_name_test,

        "nakshatra_position_deg":
            nakshatra_position_deg_test,

        "nakshatra_progress":
            nakshatra_progress_test,
    }
)


print(
    "\nCanonical Nakshatra sanity check:"
)

print(
    nakshatra_test_results
)


# ============================================================
# 7. EXPLICIT ANGULAR BOUNDARY TEST
# ============================================================

epsilon_deg = 1e-9


boundary_nakshatra_input_deg = np.array(
    [
        0.0,

        NAKSHATRA_WIDTH_DEG
        - epsilon_deg,

        NAKSHATRA_WIDTH_DEG,

        2.0 * NAKSHATRA_WIDTH_DEG
        - epsilon_deg,

        2.0 * NAKSHATRA_WIDTH_DEG,

        26.0 * NAKSHATRA_WIDTH_DEG
        - epsilon_deg,

        26.0 * NAKSHATRA_WIDTH_DEG,

        360.0
        - epsilon_deg,

        360.0,
    ]
)


boundary_nakshatra_lon_deg = (
    boundary_nakshatra_input_deg
    % 360.0
)


# ============================================================
# 8. CALCULATE BOUNDARY NAKSHATRAS
# ============================================================

boundary_nakshatra_number = (
    np.floor(
        boundary_nakshatra_lon_deg
        / NAKSHATRA_WIDTH_DEG
    )
    .astype(int)
    + 1
)


boundary_nakshatra_name = [
    NAKSHATRA_NAMES[
        number - 1
    ]
    for number in boundary_nakshatra_number
]


nakshatra_boundary_results = pd.DataFrame(
    {
        "input_sidereal_lon_deg":
            boundary_nakshatra_input_deg,

        "wrapped_sidereal_lon_deg":
            boundary_nakshatra_lon_deg,

        "nakshatra_number":
            boundary_nakshatra_number,

        "nakshatra_name":
            boundary_nakshatra_name,
    }
)


# ============================================================
# 9. DISPLAY BOUNDARY RESULTS
# ============================================================

print(
    "\nCanonical Nakshatra boundary validation:"
)

nakshatra_boundary_results

Number of Nakshatras:
27

Nakshatra angular width [deg]:
13.333333333333334

Nakshatra angular width [deg + arcmin]:
13°20′

Nakshatra number range in real test timestamps:
2 to 20

All Nakshatra numbers inside 1–27:
True

All Nakshatra progress values inside [0, 1):
True

Canonical Nakshatra sanity check:
              UTC_timestamp  lahiri_extended_deg  moon_tropical_lon_deg  \
0 2018-06-01 00:02:00+00:00            24.110343             277.274831   
1 2018-09-01 12:02:00+00:00            24.113967              48.839032   
2 2018-12-01 00:02:00+00:00            24.116863             171.499213   
3 2019-03-01 12:02:00+00:00            24.120681             284.655756   
4 2019-06-01 00:02:00+00:00            24.123458              40.332366   
5 2019-09-01 00:02:00+00:00            24.127372             180.567801   

   moon_sidereal_lon_deg  nakshatra_number   nakshatra_name  \
0             253.164488                19             Mula   
1              24.725065                

,input_sidereal_lon_deg,wrapped_sidereal_lon_deg,nakshatra_number,nakshatra_name
0,0.000000,0.000000,1,Ashwini
1,13.333333,13.333333,1,Ashwini
2,13.333333,13.333333,2,Bharani
3,26.666667,26.666667,2,Bharani
4,26.666667,26.666667,3,Krittika
5,346.666667,346.666667,26,Uttara Bhadrapada
6,346.666667,346.666667,27,Revati
7,360.000000,360.000000,27,Revati
8,360.000000,0.000000,1,Ashwini


## Full-Timeline Canonical Lahiri Ayanamsha

The canonical Lahiri convention validated on representative timestamps is
now evaluated across the complete SOLETE 5-minute timeline.

For each astronomical reference instant (`SOLETE timestamp + 2 minutes`),
Swiss Ephemeris is configured with:

- `SIDM_LAHIRI`;
- `get_ayanamsa_ex_ut()`;
- no `FLG_NONUT` flag.

The resulting extended Lahiri ayanamsha therefore includes the nutation
term required by the project's final sidereal transformation.

This full-timeline series is the canonical ayanamsha used by the final
Nakshatra and Yoga calculations.

The earlier simple `get_ayanamsa_ut()` value is no longer used for
full-timeline F4 construction.

Because the extended ayanamsha includes nutation, small short-timescale
variations are physically expected; monotonicity is therefore not
required. Instead, numerical validity and smooth adjacent changes are
checked.

In [11]:
# ============================================================
# CELL 11 — CALCULATE FULL CANONICAL EXTENDED LAHIRI AYANAMSHA
# ============================================================

# ------------------------------------------------------------
# PURPOSE
# ------------------------------------------------------------
#
# Cell 3 established the final Lahiri convention:
#
#     Swiss Ephemeris SIDM_LAHIRI
#     get_ayanamsa_ex_ut()
#     nutation included
#
# We now calculate that SAME canonical ayanamsha for every
# SOLETE t+2-minute astronomical reference timestamp.
#
# The variable:
#
#     lahiri_ayanamsha_full_deg
#
# therefore refers from this point onward to the canonical
# EXTENDED Lahiri value.
#
# No Panchang category is calculated in this cell.


# ============================================================
# 1. ENSURE STANDARD LAHIRI MODE
# ============================================================

swe.set_sid_mode(
    swe.SIDM_LAHIRI
)


# ============================================================
# 2. CONVERT FULL UTC TIMELINE TO JULIAN DAY
# ============================================================
#
# We deliberately use the same UTC -> Julian Day procedure
# validated in Cell 3.
#
# astro_reference_times contains:
#
#     SOLETE dataset timestamp + 2 minutes

lahiri_jd_full = np.empty(
    len(
        astro_reference_times
    ),
    dtype=float,
)


for i, timestamp in enumerate(
    astro_reference_times
):

    decimal_hour = (
        timestamp.hour
        + timestamp.minute / 60
        + timestamp.second / 3600
        + timestamp.microsecond / 3_600_000_000
    )

    lahiri_jd_full[i] = swe.julday(
        timestamp.year,
        timestamp.month,
        timestamp.day,
        decimal_hour,
        swe.GREG_CAL,
    )


# ============================================================
# 3. CALCULATE CANONICAL EXTENDED LAHIRI AYANAMSHA
# ============================================================
#
# get_ayanamsa_ex_ut() returns:
#
#     return_flag, ayanamsha
#
# FLG_NONUT is intentionally NOT supplied.
#
# Therefore this is the extended / nutation-inclusive Lahiri
# value selected for final F4 construction.

lahiri_ayanamsha_full_deg = np.empty(
    len(
        lahiri_jd_full
    ),
    dtype=float,
)


lahiri_ayanamsha_full_retflags = np.empty(
    len(
        lahiri_jd_full
    ),
    dtype=int,
)


for i, jd_ut in enumerate(
    lahiri_jd_full
):

    retflag, ayanamsha_extended = (
        swe.get_ayanamsa_ex_ut(
            jd_ut,
            swe.FLG_SWIEPH,
        )
    )

    lahiri_ayanamsha_full_retflags[i] = (
        retflag
    )

    lahiri_ayanamsha_full_deg[i] = (
        ayanamsha_extended
    )


# ============================================================
# 4. BASIC VALIDATION
# ============================================================

print(
    "Number of canonical ayanamsha values:"
)

print(
    len(
        lahiri_ayanamsha_full_deg
    )
)


print(
    "\nAll canonical ayanamsha values finite:"
)

print(
    np.isfinite(
        lahiri_ayanamsha_full_deg
    ).all()
)


print(
    "\nCanonical extended Lahiri range [deg]:"
)

print(
    lahiri_ayanamsha_full_deg.min(),
    "to",
    lahiri_ayanamsha_full_deg.max(),
)


print(
    "\nFirst-to-last canonical Lahiri change [deg]:"
)

print(
    lahiri_ayanamsha_full_deg[-1]
    -
    lahiri_ayanamsha_full_deg[0]
)


print(
    "\nUnique extended-ayanamsha return flags:"
)

print(
    np.unique(
        lahiri_ayanamsha_full_retflags
    )
)


# ============================================================
# 5. VERIFY AGAINST THE SIX CANONICAL CELL-3 VALUES
# ============================================================
#
# The full calculation must reproduce the six representative
# extended-Lahiri values established in Cell 3.

lahiri_full_series = pd.Series(
    lahiri_ayanamsha_full_deg,
    index=astro_reference_times,
    name="lahiri_ayanamsha_extended_deg",
)


recovered_test_ayanamsha = (
    lahiri_full_series.loc[
        ayanamsha_test_times
    ]
    .to_numpy()
)


ayanamsha_reproduction_error_deg = np.abs(
    recovered_test_ayanamsha
    -
    lahiri_ayanamsha_test_deg
)


print(
    "\nMaximum reproduction error against canonical "
    "Cell 3 [deg]:"
)

print(
    ayanamsha_reproduction_error_deg.max()
)


# ============================================================
# 6. VERIFY SMOOTH ADJACENT EVOLUTION
# ============================================================
#
# IMPORTANT:
#
# The extended ayanamsha includes nutation.
#
# Therefore it is not required to increase monotonically.
#
# We instead inspect the maximum absolute change between
# adjacent 5-minute astronomical reference timestamps.

ayanamsha_step_change_deg = np.diff(
    lahiri_ayanamsha_full_deg
)


print(
    "\nMaximum absolute 5-minute canonical "
    "ayanamsha change [deg]:"
)

print(
    np.abs(
        ayanamsha_step_change_deg
    ).max()
)


# ============================================================
# 7. BUILD CANONICAL AUDIT TABLE
# ============================================================
#
# The table remains indexed by the ORIGINAL SOLETE timestamp
# t, even though each ayanamsha value was evaluated at:
#
#     t + 2 minutes
#
# This preserves alignment with the forecasting dataset.

df_ayanamsha = pd.DataFrame(
    {
        "lahiri_ayanamsha_extended_deg":
            lahiri_ayanamsha_full_deg,
    },
    index=df_raw.index,
)


# ============================================================
# 8. TABLE VALIDATION
# ============================================================

print(
    "\nCanonical ayanamsha table shape:"
)

print(
    df_ayanamsha.shape
)


print(
    "\nCanonical ayanamsha missing values:"
)

print(
    df_ayanamsha.isna().sum()
)


# ============================================================
# 9. DISPLAY SAMPLE
# ============================================================

print(
    "\nSample canonical Lahiri ayanamsha values:"
)

df_ayanamsha.head(12)

Number of canonical ayanamsha values:
131617

All canonical ayanamsha values finite:
True

Canonical extended Lahiri range [deg]:
24.110343268621794 to 24.12737225615406

First-to-last canonical Lahiri change [deg]:
0.017028748020788242

Unique extended-ayanamsha return flags:
[2]

Maximum reproduction error against canonical Cell 3 [deg]:
0.0

Maximum absolute 5-minute canonical ayanamsha change [deg]:
3.060621871497915e-07

Canonical ayanamsha table shape:
(131617, 1)

Canonical ayanamsha missing values:
lahiri_ayanamsha_extended_deg    0
dtype: int64

Sample canonical Lahiri ayanamsha values:


,lahiri_ayanamsha_extended_deg
Timestamp,
2018-06-01 00:00:00+00:00,24.110343
2018-06-01 00:05:00+00:00,24.110343
2018-06-01 00:10:00+00:00,24.110344
2018-06-01 00:15:00+00:00,24.110344
2018-06-01 00:20:00+00:00,24.110344
2018-06-01 00:25:00+00:00,24.110344
2018-06-01 00:30:00+00:00,24.110345
2018-06-01 00:35:00+00:00,24.110345
2018-06-01 00:40:00+00:00,24.110345


## Full-Timeline Canonical Nakshatra Generation

The canonical extended Lahiri ayanamsha has now been calculated for all
131,617 SOLETE astronomical reference timestamps.

For every observation, the final Lahiri-aligned sidereal lunar longitude
is constructed as:

`sidereal Moon longitude = Skyfield tropical Moon longitude - extended Lahiri ayanamsha`

wrapped into `[0°, 360°)`.

The resulting longitude is divided into 27 equal Nakshatra sectors of:

`13°20′`

This cell therefore produces the **canonical full-timeline Nakshatra
representation** used by F4.

The earlier simple-ayanamsha Nakshatra assignments are no longer part of
the normal feature-generation path.

Continuous sidereal longitude and within-Nakshatra progress are retained
for audit and reproducibility; the later raw F4 model table uses the
discrete Nakshatra state.

In [12]:
# ============================================================
# CELL 12 — GENERATE FULL CANONICAL NAKSHATRA TIMELINE
# ============================================================

# ------------------------------------------------------------
# PURPOSE
# ------------------------------------------------------------
#
# Cell 10 validated the Nakshatra definition using the
# canonical extended Lahiri transformation.
#
# Cell 11 generated the same extended Lahiri ayanamsha for
# every SOLETE astronomical reference timestamp.
#
# We now construct the FINAL full-timeline Nakshatra values.
#
# Canonical sidereal convention:
#
#     Skyfield apparent tropical lunar longitude
#         -
#     extended Lahiri ayanamsha
#
# Nakshatra is AYANAMSHA-DEPENDENT.


# ============================================================
# 1. CONSTRUCT CANONICAL SIDEREAL LUNAR LONGITUDE
# ============================================================

moon_sidereal_longitude_full_deg = (
    (
        moon_tropical_longitude_full_deg
        -
        lahiri_ayanamsha_full_deg
    )
    % 360.0
)


# ============================================================
# 2. CALCULATE NAKSHATRA NUMBER
# ============================================================

nakshatra_number_full = (
    np.floor(
        moon_sidereal_longitude_full_deg
        / NAKSHATRA_WIDTH_DEG
    )
    .astype(int)
    + 1
)


# ============================================================
# 3. ASSIGN NAKSHATRA NAME
# ============================================================

nakshatra_name_full = [
    NAKSHATRA_NAMES[
        number - 1
    ]
    for number in nakshatra_number_full
]


# ============================================================
# 4. CALCULATE POSITION WITHIN CURRENT NAKSHATRA
# ============================================================
#
# Audit / diagnostic quantities:
#
#     position:
#         0 <= value < 13⅓ degrees
#
#     progress:
#         0 <= value < 1

nakshatra_position_full_deg = (
    moon_sidereal_longitude_full_deg
    % NAKSHATRA_WIDTH_DEG
)


nakshatra_progress_full = (
    nakshatra_position_full_deg
    / NAKSHATRA_WIDTH_DEG
)


# ============================================================
# 5. BUILD CANONICAL NAKSHATRA TABLE
# ============================================================

df_nakshatra = pd.DataFrame(
    {
        # ---------------------------------------------
        # Canonical Lahiri reference
        # ---------------------------------------------
        "lahiri_ayanamsha_extended_deg":
            lahiri_ayanamsha_full_deg,

        # ---------------------------------------------
        # Astronomical audit quantities
        # ---------------------------------------------
        "moon_tropical_longitude_deg":
            moon_tropical_longitude_full_deg,

        "moon_sidereal_longitude_deg":
            moon_sidereal_longitude_full_deg,

        # ---------------------------------------------
        # Panchang quantity
        # ---------------------------------------------
        "nakshatra_number":
            nakshatra_number_full,

        "nakshatra_name":
            nakshatra_name_full,

        # ---------------------------------------------
        # Audit / diagnostic quantities
        # ---------------------------------------------
        "nakshatra_position_deg":
            nakshatra_position_full_deg,

        "nakshatra_progress":
            nakshatra_progress_full,
    },
    index=df_raw.index,
)


# ============================================================
# 6. BASIC VALIDATION
# ============================================================

print(
    "Canonical Nakshatra table shape:"
)

print(
    df_nakshatra.shape
)


print(
    "\nMissing values:"
)

print(
    df_nakshatra.isna().sum()
)


print(
    "\nCanonical sidereal lunar-longitude range [deg]:"
)

print(
    df_nakshatra[
        "moon_sidereal_longitude_deg"
    ].min(),
    "to",
    df_nakshatra[
        "moon_sidereal_longitude_deg"
    ].max(),
)


print(
    "\nNakshatra number range:"
)

print(
    df_nakshatra[
        "nakshatra_number"
    ].min(),
    "to",
    df_nakshatra[
        "nakshatra_number"
    ].max(),
)


print(
    "\nUnique Nakshatra numbers:"
)

print(
    df_nakshatra[
        "nakshatra_number"
    ].nunique()
)


print(
    "\nUnique Nakshatra names:"
)

print(
    df_nakshatra[
        "nakshatra_name"
    ].nunique()
)


print(
    "\nNakshatra-progress range:"
)

print(
    df_nakshatra[
        "nakshatra_progress"
    ].min(),
    "to",
    df_nakshatra[
        "nakshatra_progress"
    ].max(),
)


# ============================================================
# 7. CHECK ALL 27 NAKSHATRAS ARE PRESENT
# ============================================================

expected_nakshatras = set(
    range(
        1,
        28,
    )
)


observed_nakshatras = set(
    df_nakshatra[
        "nakshatra_number"
    ].unique()
)


missing_nakshatras = sorted(
    expected_nakshatras
    -
    observed_nakshatras
)


print(
    "\nMissing Nakshatra numbers:"
)

print(
    missing_nakshatras
)


# ============================================================
# 8. VERIFY SIX CANONICAL CELL-10 TEST TIMESTAMPS
# ============================================================
#
# df_nakshatra is indexed by the original SOLETE timestamp:
#
#     t
#
# Cell 10 test timestamps represent:
#
#     t + 2 minutes
#
# Convert them back to their corresponding dataset rows before
# performing the reproduction check.

nakshatra_test_dataset_times = (
    ayanamsha_test_times
    -
    pd.Timedelta(
        minutes=2
    )
)


recovered_nakshatra_numbers = (
    df_nakshatra.loc[
        nakshatra_test_dataset_times,
        "nakshatra_number"
    ]
    .to_numpy()
)


nakshatra_reproduction_mismatch_count = (
    recovered_nakshatra_numbers
    !=
    nakshatra_number_test
).sum()


print(
    "\nCanonical Nakshatra reproduction mismatch "
    "against Cell 10:"
)

print(
    nakshatra_reproduction_mismatch_count
)


print(
    "\nRecovered Nakshatra numbers:"
)

print(
    recovered_nakshatra_numbers
)


print(
    "\nExpected Cell 10 Nakshatra numbers:"
)

print(
    nakshatra_number_test
)


# ============================================================
# 9. OBSERVATION COUNTS BY NAKSHATRA
# ============================================================

nakshatra_counts = (
    df_nakshatra[
        [
            "nakshatra_number",
            "nakshatra_name",
        ]
    ]
    .value_counts()
    .sort_index()
)


print(
    "\nObservation counts by canonical Nakshatra:"
)

print(
    nakshatra_counts
)


# ============================================================
# 10. DISPLAY SAMPLE
# ============================================================

print(
    "\nSample canonical Nakshatra table:"
)

df_nakshatra.head(12)

Canonical Nakshatra table shape:
(131617, 7)

Missing values:
lahiri_ayanamsha_extended_deg    0
moon_tropical_longitude_deg      0
moon_sidereal_longitude_deg      0
nakshatra_number                 0
nakshatra_name                   0
nakshatra_position_deg           0
nakshatra_progress               0
dtype: int64

Canonical sidereal lunar-longitude range [deg]:
9.34330612629708e-05 to 359.99711128054196

Nakshatra number range:
1 to 27

Unique Nakshatra numbers:
27

Unique Nakshatra names:
27

Nakshatra-progress range:
6.50378099535942e-06 to 0.9999791455604392

Missing Nakshatra numbers:
[]

Canonical Nakshatra reproduction mismatch against Cell 10:
0

Recovered Nakshatra numbers:
[19  2 12 20  2 12]

Expected Cell 10 Nakshatra numbers:
[19  2 12 20  2 12]

Observation counts by canonical Nakshatra:
nakshatra_number  nakshatra_name   
1                 Ashwini              5076
2                 Bharani              4957
3                 Krittika             4840
4              

,lahiri_ayanamsha_extended_deg,moon_tropical_longitude_deg,moon_sidereal_longitude_deg,nakshatra_number,nakshatra_name,nakshatra_position_deg,nakshatra_progress
Timestamp,,,,,,,
2018-06-01 00:00:00+00:00,24.110343,277.274831,253.164488,19,Mula,13.164488,0.987337
2018-06-01 00:05:00+00:00,24.110343,277.316238,253.205895,19,Mula,13.205895,0.990442
2018-06-01 00:10:00+00:00,24.110344,277.357643,253.247300,19,Mula,13.247300,0.993547
2018-06-01 00:15:00+00:00,24.110344,277.399047,253.288703,19,Mula,13.288703,0.996653
2018-06-01 00:20:00+00:00,24.110344,277.440449,253.330105,19,Mula,13.330105,0.999758
2018-06-01 00:25:00+00:00,24.110344,277.481850,253.371506,20,Purva Ashadha,0.038173,0.002863
2018-06-01 00:30:00+00:00,24.110345,277.523250,253.412905,20,Purva Ashadha,0.079572,0.005968
2018-06-01 00:35:00+00:00,24.110345,277.564648,253.454303,20,Purva Ashadha,0.120969,0.009073
2018-06-01 00:40:00+00:00,24.110345,277.606044,253.495699,20,Purva Ashadha,0.162366,0.012177


## Yoga Construction and Boundary Validation

Panchang Yoga (Nitya Yoga) is the second F4 quantity that depends on the
adopted sidereal reference.

It is constructed from the sum of the canonical Lahiri-aligned sidereal
longitudes of the Sun and Moon:

`Yoga angle = (sidereal Sun longitude + sidereal Moon longitude) mod 360°`

where each sidereal longitude is defined as:

`Skyfield apparent tropical longitude - extended Lahiri ayanamsha`

The resulting 360-degree Yoga cycle is divided into 27 equal sectors:

`360° / 27 = 13°20′`

Unlike Tithi, Paksha and Karana, the common ayanamsha does not cancel
from Yoga.

If `A` is the canonical extended Lahiri ayanamsha:

`Moon_sidereal = Moon_tropical - A`

`Sun_sidereal = Sun_tropical - A`

therefore:

`Yoga angle = (Moon_tropical + Sun_tropical - 2A) mod 360°`

Yoga is consequently explicitly ayanamsha-dependent.

This cell validates the Yoga construction on the six representative
timestamps and tests its angular sector boundaries before generating
the full-timeline feature.

In [13]:
# ============================================================
# CELL 13 — CANONICAL YOGA SANITY AND BOUNDARY VALIDATION
# ============================================================

# ------------------------------------------------------------
# PURPOSE
# ------------------------------------------------------------
#
# Yoga is an AYANAMSHA-DEPENDENT Panchang quantity.
#
# This cell uses the canonical sidereal longitudes established
# in Cell 4:
#
#     Skyfield apparent tropical longitude
#         -
#     extended Lahiri ayanamsha
#
# Yoga is defined as:
#
#     (
#         sidereal Sun longitude
#         +
#         sidereal Moon longitude
#     ) mod 360
#
# We validate:
#
#     1. the 27 traditional Yoga names;
#     2. six representative real timestamps;
#     3. the expected -2 * ayanamsha relationship;
#     4. angular sector boundaries.
#
# No full-timeline Yoga feature is generated here.


# ============================================================
# 1. DEFINE YOGA ANGULAR WIDTH
# ============================================================

YOGA_COUNT = 27


YOGA_WIDTH_DEG = (
    360.0
    / YOGA_COUNT
)


print("Number of Yogas:")
print(
    YOGA_COUNT
)


print("\nYoga angular width [deg]:")
print(
    YOGA_WIDTH_DEG
)


print("\nYoga angular width [deg + arcmin]:")
print(
    "13°20′"
)


# ============================================================
# 2. DEFINE THE 27 NITYA YOGA NAMES
# ============================================================
#
# English transliterations differ slightly across references.
#
# One fixed spelling convention is retained consistently
# throughout this project.

YOGA_NAMES = [
    "Vishkambha",
    "Priti",
    "Ayushman",
    "Saubhagya",
    "Shobhana",
    "Atiganda",
    "Sukarma",
    "Dhriti",
    "Shula",
    "Ganda",
    "Vriddhi",
    "Dhruva",
    "Vyaghata",
    "Harshana",
    "Vajra",
    "Siddhi",
    "Vyatipata",
    "Variyana",
    "Parigha",
    "Shiva",
    "Siddha",
    "Sadhya",
    "Shubha",
    "Shukla",
    "Brahma",
    "Indra",
    "Vaidhriti",
]


assert len(
    YOGA_NAMES
) == YOGA_COUNT


# ============================================================
# 3. CALCULATE YOGA ANGLE FOR SIX REAL TEST TIMESTAMPS
# ============================================================
#
# These variables now contain canonical extended-Lahiri
# sidereal longitudes from Cell 4:
#
#     sun_sidereal_longitude_deg
#     moon_sidereal_longitude_deg

yoga_angle_test_deg = (
    (
        sun_sidereal_longitude_deg
        +
        moon_sidereal_longitude_deg
    )
    % 360.0
)


# ============================================================
# 4. VERIFY THE CANONICAL AYANAMSHA RELATIONSHIP
# ============================================================
#
# If:
#
#     Moon_sidereal = Moon_tropical - A
#     Sun_sidereal  = Sun_tropical  - A
#
# then:
#
#     Yoga
#         =
#     Moon_tropical
#         +
#     Sun_tropical
#         -
#     2A
#
# modulo 360 degrees.
#
# Here A is the canonical EXTENDED Lahiri ayanamsha.

yoga_angle_expected_test_deg = (
    (
        moon_tropical_longitude_deg
        +
        sun_tropical_longitude_deg
        -
        2.0 * lahiri_ayanamsha_test_deg
    )
    % 360.0
)


# ------------------------------------------------------------
# Use the smallest circular angular difference so the
# 0° / 360° wrap does not create a false error.
# ------------------------------------------------------------

yoga_ayanamsha_relation_error_deg = np.abs(
    (
        yoga_angle_test_deg
        -
        yoga_angle_expected_test_deg
        +
        180.0
    )
    % 360.0
    -
    180.0
)


print(
    "\nMaximum canonical Yoga "
    "ayanamsha-relation error [deg]:"
)

print(
    yoga_ayanamsha_relation_error_deg.max()
)


# ============================================================
# 5. CALCULATE YOGA NUMBER
# ============================================================

yoga_number_test = (
    np.floor(
        yoga_angle_test_deg
        / YOGA_WIDTH_DEG
    )
    .astype(int)
    + 1
)


# ============================================================
# 6. ASSIGN YOGA NAME
# ============================================================

yoga_name_test = [
    YOGA_NAMES[
        number - 1
    ]
    for number in yoga_number_test
]


# ============================================================
# 7. CALCULATE POSITION WITHIN CURRENT YOGA
# ============================================================
#
# Audit / diagnostic quantities:
#
#     position:
#         0 <= value < 13⅓ degrees
#
#     progress:
#         0 <= value < 1

yoga_position_deg_test = (
    yoga_angle_test_deg
    % YOGA_WIDTH_DEG
)


yoga_progress_test = (
    yoga_position_deg_test
    / YOGA_WIDTH_DEG
)


# ============================================================
# 8. BASIC VALIDATION
# ============================================================

print(
    "\nYoga number range in real test timestamps:"
)

print(
    yoga_number_test.min(),
    "to",
    yoga_number_test.max(),
)


print(
    "\nAll Yoga numbers inside 1–27:"
)

print(
    (
        (yoga_number_test >= 1)
        &
        (yoga_number_test <= 27)
    ).all()
)


print(
    "\nAll Yoga progress values inside [0, 1):"
)

print(
    (
        (yoga_progress_test >= 0)
        &
        (yoga_progress_test < 1)
    ).all()
)


# ============================================================
# 9. BUILD SIX-TIMESTAMP VALIDATION TABLE
# ============================================================

yoga_test_results = pd.DataFrame(
    {
        "UTC_timestamp":
            ayanamsha_test_times,

        "lahiri_extended_deg":
            lahiri_ayanamsha_test_deg,

        "sun_sidereal_lon_deg":
            sun_sidereal_longitude_deg,

        "moon_sidereal_lon_deg":
            moon_sidereal_longitude_deg,

        "yoga_angle_deg":
            yoga_angle_test_deg,

        "yoga_number":
            yoga_number_test,

        "yoga_name":
            yoga_name_test,

        "yoga_position_deg":
            yoga_position_deg_test,

        "yoga_progress":
            yoga_progress_test,
    }
)


print(
    "\nCanonical Yoga sanity check:"
)

print(
    yoga_test_results
)


# ============================================================
# 10. EXPLICIT ANGULAR BOUNDARY TEST
# ============================================================

epsilon_deg = 1e-9


boundary_yoga_input_deg = np.array(
    [
        0.0,

        YOGA_WIDTH_DEG
        - epsilon_deg,

        YOGA_WIDTH_DEG,

        2.0 * YOGA_WIDTH_DEG
        - epsilon_deg,

        2.0 * YOGA_WIDTH_DEG,

        26.0 * YOGA_WIDTH_DEG
        - epsilon_deg,

        26.0 * YOGA_WIDTH_DEG,

        360.0
        - epsilon_deg,

        360.0,
    ]
)


boundary_yoga_angle_deg = (
    boundary_yoga_input_deg
    % 360.0
)


# ============================================================
# 11. CALCULATE BOUNDARY YOGAS
# ============================================================

boundary_yoga_number = (
    np.floor(
        boundary_yoga_angle_deg
        / YOGA_WIDTH_DEG
    )
    .astype(int)
    + 1
)


boundary_yoga_name = [
    YOGA_NAMES[
        number - 1
    ]
    for number in boundary_yoga_number
]


yoga_boundary_results = pd.DataFrame(
    {
        "input_yoga_angle_deg":
            boundary_yoga_input_deg,

        "wrapped_yoga_angle_deg":
            boundary_yoga_angle_deg,

        "yoga_number":
            boundary_yoga_number,

        "yoga_name":
            boundary_yoga_name,
    }
)


# ============================================================
# 12. DISPLAY BOUNDARY RESULTS
# ============================================================

print(
    "\nCanonical Yoga boundary validation:"
)

yoga_boundary_results

Number of Yogas:
27

Yoga angular width [deg]:
13.333333333333334

Yoga angular width [deg + arcmin]:
13°20′

Maximum canonical Yoga ayanamsha-relation error [deg]:
5.684341886080802e-14

Yoga number range in real test timestamps:
1 to 23

All Yoga numbers inside 1–27:
True

All Yoga progress values inside [0, 1):
True

Canonical Yoga sanity check:
              UTC_timestamp  lahiri_extended_deg  sun_sidereal_lon_deg  \
0 2018-06-01 00:02:00+00:00            24.110343             46.362341   
1 2018-09-01 12:02:00+00:00            24.113967            134.890651   
2 2018-12-01 00:02:00+00:00            24.116863            224.609083   
3 2019-03-01 12:02:00+00:00            24.120681            316.485264   
4 2019-06-01 00:02:00+00:00            24.123458             46.122044   
5 2019-09-01 00:02:00+00:00            24.127372            134.160767   

   moon_sidereal_lon_deg  yoga_angle_deg  yoga_number   yoga_name  \
0             253.164488      299.526829           23      Sh

,input_yoga_angle_deg,wrapped_yoga_angle_deg,yoga_number,yoga_name
0,0.000000,0.000000,1,Vishkambha
1,13.333333,13.333333,1,Vishkambha
2,13.333333,13.333333,2,Priti
3,26.666667,26.666667,2,Priti
4,26.666667,26.666667,3,Ayushman
5,346.666667,346.666667,26,Indra
6,346.666667,346.666667,27,Vaidhriti
7,360.000000,360.000000,27,Vaidhriti
8,360.000000,0.000000,1,Vishkambha


## Full-Timeline Canonical Yoga Generation

The canonical Yoga formulation has now been validated using the
extended Lahiri sidereal transformation.

For every SOLETE astronomical reference timestamp:

1. the apparent geocentric tropical solar longitude is obtained from
   Skyfield + JPL DE440s;
2. the canonical extended Lahiri ayanamsha is subtracted;
3. the resulting sidereal solar longitude is combined with the already
   constructed sidereal lunar longitude;
4. the wrapped sum is divided into 27 equal Yoga sectors of `13°20′`.

The resulting table is the **canonical full-timeline Yoga
representation** used by F4.

The earlier simple-ayanamsha Yoga assignments are no longer part of the
normal feature-generation path.

Continuous Yoga angle and within-Yoga progress are retained only for
audit and reproducibility. The later raw F4 model table uses the
discrete Yoga state.

In [14]:
# ============================================================
# CELL 14 — GENERATE FULL CANONICAL YOGA TIMELINE
# ============================================================

# ------------------------------------------------------------
# PURPOSE
# ------------------------------------------------------------
#
# Cell 13 validated Yoga using the canonical extended Lahiri
# transformation.
#
# We now calculate Yoga for all 131,617 SOLETE observations.
#
# Canonical sidereal convention:
#
#     Skyfield apparent tropical longitude
#         -
#     extended Lahiri ayanamsha
#
# Yoga is AYANAMSHA-DEPENDENT.


# ============================================================
# 1. CONSTRUCT CANONICAL SIDEREAL SOLAR LONGITUDE
# ============================================================

sun_sidereal_longitude_full_deg = (
    (
        sun_tropical_longitude_full_deg
        -
        lahiri_ayanamsha_full_deg
    )
    % 360.0
)


# ============================================================
# 2. CALCULATE CANONICAL FULL YOGA ANGLE
# ============================================================

yoga_angle_full_deg = (
    (
        sun_sidereal_longitude_full_deg
        +
        moon_sidereal_longitude_full_deg
    )
    % 360.0
)


# ============================================================
# 3. CALCULATE YOGA NUMBER
# ============================================================

yoga_number_full = (
    np.floor(
        yoga_angle_full_deg
        / YOGA_WIDTH_DEG
    )
    .astype(int)
    + 1
)


# ============================================================
# 4. ASSIGN YOGA NAME
# ============================================================

yoga_name_full = [
    YOGA_NAMES[
        number - 1
    ]
    for number in yoga_number_full
]


# ============================================================
# 5. CALCULATE POSITION WITHIN CURRENT YOGA
# ============================================================
#
# Audit / diagnostic quantities:
#
#     position:
#         0 <= value < 13⅓ degrees
#
#     progress:
#         0 <= value < 1

yoga_position_full_deg = (
    yoga_angle_full_deg
    % YOGA_WIDTH_DEG
)


yoga_progress_full = (
    yoga_position_full_deg
    / YOGA_WIDTH_DEG
)


# ============================================================
# 6. BUILD CANONICAL YOGA TABLE
# ============================================================

df_yoga = pd.DataFrame(
    {
        # ---------------------------------------------
        # Canonical Lahiri reference
        # ---------------------------------------------
        "lahiri_ayanamsha_extended_deg":
            lahiri_ayanamsha_full_deg,

        # ---------------------------------------------
        # Sidereal astronomical audit quantities
        # ---------------------------------------------
        "sun_sidereal_longitude_deg":
            sun_sidereal_longitude_full_deg,

        "moon_sidereal_longitude_deg":
            moon_sidereal_longitude_full_deg,

        # ---------------------------------------------
        # Continuous Yoga audit quantity
        # ---------------------------------------------
        "yoga_angle_deg":
            yoga_angle_full_deg,

        # ---------------------------------------------
        # Panchang quantity
        # ---------------------------------------------
        "yoga_number":
            yoga_number_full,

        "yoga_name":
            yoga_name_full,

        # ---------------------------------------------
        # Audit / diagnostic quantities
        # ---------------------------------------------
        "yoga_position_deg":
            yoga_position_full_deg,

        "yoga_progress":
            yoga_progress_full,
    },
    index=df_raw.index,
)


# ============================================================
# 7. BASIC VALIDATION
# ============================================================

print(
    "Canonical Yoga table shape:"
)

print(
    df_yoga.shape
)


print(
    "\nMissing values:"
)

print(
    df_yoga.isna().sum()
)


print(
    "\nCanonical sidereal solar-longitude range [deg]:"
)

print(
    df_yoga[
        "sun_sidereal_longitude_deg"
    ].min(),
    "to",
    df_yoga[
        "sun_sidereal_longitude_deg"
    ].max(),
)


print(
    "\nCanonical Yoga-angle range [deg]:"
)

print(
    df_yoga[
        "yoga_angle_deg"
    ].min(),
    "to",
    df_yoga[
        "yoga_angle_deg"
    ].max(),
)


print(
    "\nYoga number range:"
)

print(
    df_yoga[
        "yoga_number"
    ].min(),
    "to",
    df_yoga[
        "yoga_number"
    ].max(),
)


print(
    "\nUnique Yoga numbers:"
)

print(
    df_yoga[
        "yoga_number"
    ].nunique()
)


print(
    "\nUnique Yoga names:"
)

print(
    df_yoga[
        "yoga_name"
    ].nunique()
)


print(
    "\nYoga-progress range:"
)

print(
    df_yoga[
        "yoga_progress"
    ].min(),
    "to",
    df_yoga[
        "yoga_progress"
    ].max(),
)


# ============================================================
# 8. CHECK ALL 27 YOGAS ARE PRESENT
# ============================================================

expected_yogas = set(
    range(
        1,
        28,
    )
)


observed_yogas = set(
    df_yoga[
        "yoga_number"
    ].unique()
)


missing_yogas = sorted(
    expected_yogas
    -
    observed_yogas
)


print(
    "\nMissing Yoga numbers:"
)

print(
    missing_yogas
)


# ============================================================
# 9. VERIFY FULL CANONICAL AYANAMSHA RELATIONSHIP
# ============================================================
#
# Independently reconstruct Yoga from tropical longitudes:
#
#     Moon_tropical
#         +
#     Sun_tropical
#         -
#     2 * extended Lahiri ayanamsha
#
# This should reproduce yoga_angle_full_deg to floating-point
# precision.

yoga_expected_full_deg = (
    (
        moon_tropical_longitude_full_deg
        +
        sun_tropical_longitude_full_deg
        -
        2.0 * lahiri_ayanamsha_full_deg
    )
    % 360.0
)


yoga_relation_error_full_deg = np.abs(
    (
        yoga_angle_full_deg
        -
        yoga_expected_full_deg
        +
        180.0
    )
    % 360.0
    -
    180.0
)


print(
    "\nMaximum full-timeline canonical Yoga "
    "ayanamsha-relation error [deg]:"
)

print(
    yoga_relation_error_full_deg.max()
)


# ============================================================
# 10. VERIFY SIX CANONICAL CELL-13 TEST TIMESTAMPS
# ============================================================
#
# df_yoga uses original SOLETE timestamps:
#
#     t
#
# Cell 13 validation timestamps represent:
#
#     t + 2 minutes
#
# Convert them back before indexing.

yoga_test_dataset_times = (
    ayanamsha_test_times
    -
    pd.Timedelta(
        minutes=2
    )
)


recovered_yoga_numbers = (
    df_yoga.loc[
        yoga_test_dataset_times,
        "yoga_number"
    ]
    .to_numpy()
)


yoga_reproduction_mismatch_count = (
    recovered_yoga_numbers
    !=
    yoga_number_test
).sum()


print(
    "\nCanonical Yoga reproduction mismatch "
    "against Cell 13:"
)

print(
    yoga_reproduction_mismatch_count
)


print(
    "\nRecovered Yoga numbers:"
)

print(
    recovered_yoga_numbers
)


print(
    "\nExpected Cell 13 Yoga numbers:"
)

print(
    yoga_number_test
)


# ============================================================
# 11. OBSERVATION COUNTS BY YOGA
# ============================================================

yoga_counts = (
    df_yoga[
        [
            "yoga_number",
            "yoga_name",
        ]
    ]
    .value_counts()
    .sort_index()
)


print(
    "\nObservation counts by canonical Yoga:"
)

print(
    yoga_counts
)


# ============================================================
# 12. DISPLAY SAMPLE
# ============================================================

print(
    "\nSample canonical Yoga table:"
)

df_yoga.head(12)

Canonical Yoga table shape:
(131617, 8)

Missing values:
lahiri_ayanamsha_extended_deg    0
sun_sidereal_longitude_deg       0
moon_sidereal_longitude_deg      0
yoga_angle_deg                   0
yoga_number                      0
yoga_name                        0
yoga_position_deg                0
yoga_progress                    0
dtype: int64

Canonical sidereal solar-longitude range [deg]:
0.0019983042624325265 to 359.99859667670586

Canonical Yoga-angle range [deg]:
0.0004052654750239526 to 359.9919375410051

Yoga number range:
1 to 27

Unique Yoga numbers:
27

Unique Yoga names:
27

Yoga-progress range:
4.102734037703115e-06 to 0.9999965351155246

Missing Yoga numbers:
[]

Maximum full-timeline canonical Yoga ayanamsha-relation error [deg]:
1.1368683772161603e-13

Canonical Yoga reproduction mismatch against Cell 13:
0

Recovered Yoga numbers:
[23 12  1 17  5 22]

Expected Cell 13 Yoga numbers:
[23 12  1 17  5 22]

Observation counts by canonical Yoga:
yoga_number  yoga_name 
1

,lahiri_ayanamsha_extended_deg,sun_sidereal_longitude_deg,moon_sidereal_longitude_deg,yoga_angle_deg,yoga_number,yoga_name,yoga_position_deg,yoga_progress
Timestamp,,,,,,,,
2018-06-01 00:00:00+00:00,24.110343,46.362341,253.164488,299.526829,23,Shubha,6.193496,0.464512
2018-06-01 00:05:00+00:00,24.110343,46.365669,253.205895,299.571563,23,Shubha,6.238230,0.467867
2018-06-01 00:10:00+00:00,24.110344,46.368996,253.247300,299.616295,23,Shubha,6.282962,0.471222
2018-06-01 00:15:00+00:00,24.110344,46.372323,253.288703,299.661026,23,Shubha,6.327693,0.474577
2018-06-01 00:20:00+00:00,24.110344,46.375650,253.330105,299.705755,23,Shubha,6.372422,0.477932
2018-06-01 00:25:00+00:00,24.110344,46.378977,253.371506,299.750483,23,Shubha,6.417150,0.481286
2018-06-01 00:30:00+00:00,24.110345,46.382304,253.412905,299.795209,23,Shubha,6.461876,0.484641
2018-06-01 00:35:00+00:00,24.110345,46.385631,253.454303,299.839934,23,Shubha,6.506601,0.487995
2018-06-01 00:40:00+00:00,24.110345,46.388958,253.495699,299.884657,23,Shubha,6.551324,0.491349


## Consolidating the Canonical Panchang Feature Set

The five initially selected Panchang quantities have now been
constructed and validated across the complete SOLETE timeline:

- Tithi
- Paksha
- Karana
- Nakshatra
- Yoga

The final implementation uses two distinct representations.

### 1. Panchang audit table

The audit table preserves the astronomical and intermediate quantities
needed for:

- reproducibility;
- boundary inspection;
- debugging;
- methodological validation;
- interpretation.

It includes continuous quantities such as Moon-Sun phase, Lahiri
sidereal longitude, Yoga angle and within-sector progress.

These quantities are **not automatically forecasting inputs**.

### 2. Raw F4 model-feature table

The raw F4 table contains only the structured Panchang states intended
for the controlled feature-ablation experiment:

- Tithi;
- Paksha;
- Karana;
- Nakshatra;
- Yoga.

Nakshatra and Yoga in this table use the project's final canonical
sidereal convention:

`Skyfield apparent tropical longitude - extended Lahiri ayanamsha`

where the extended Lahiri ayanamsha is obtained from Swiss Ephemeris
`SIDM_LAHIRI` using `get_ayanamsa_ex_ut()`.

Continuous astronomical quantities are deliberately excluded from the
raw F4 model table because related continuous Sun-Moon geometry already
belongs to F3.

Thus, the intended comparison remains:

**F3 = continuous astronomical representation**

versus

**F4 = structured Panchang representation**

All five raw F4 variables remain categorical. Model-specific encoding
will be performed later inside the forecasting pipeline rather than in
this feature-engineering notebook.

In [15]:
# ============================================================
# CELL 15 — CONSOLIDATE CANONICAL PANCHANG TABLES
# ============================================================

# ------------------------------------------------------------
# PURPOSE
# ------------------------------------------------------------
#
# The full canonical Panchang construction is now complete.
#
# We create:
#
#     1. df_panchang_audit
#        -> scientific / reproducibility quantities
#
#     2. df_panchang_f4_raw
#        -> categorical variables intended for the F4
#           forecasting feature family
#
# IMPORTANT:
#
# Nakshatra and Yoga are already canonical because
# df_nakshatra and df_yoga were generated using the extended
# Lahiri ayanamsha in Cells 12 and 14.
#
# There is therefore no provisional F4 table in the normal
# notebook path anymore.


# ============================================================
# 1. VERIFY ALL PANCHANG TABLES SHARE THE SOLETE INDEX
# ============================================================

index_checks = {
    "Tithi":
        df_tithi.index.equals(
            df_raw.index
        ),

    "Karana":
        df_karana.index.equals(
            df_raw.index
        ),

    "Nakshatra":
        df_nakshatra.index.equals(
            df_raw.index
        ),

    "Yoga":
        df_yoga.index.equals(
            df_raw.index
        ),
}


print(
    "Index alignment checks:"
)

print(
    index_checks
)


assert all(
    index_checks.values()
)


# ============================================================
# 2. BUILD CANONICAL PANCHANG AUDIT TABLE
# ============================================================
#
# This table contains both:
#
#     Panchang states
#
# and:
#
#     continuous astronomical / intermediate quantities.
#
# It exists for validation and reproducibility.
#
# It is NOT the F4 model-input table.

df_panchang_audit = pd.DataFrame(
    {
        # ----------------------------------------------------
        # Canonical Lahiri reference
        # ----------------------------------------------------
        "lahiri_ayanamsha_extended_deg":
            lahiri_ayanamsha_full_deg,

        # ----------------------------------------------------
        # Tropical astronomical quantities
        # ----------------------------------------------------
        "sun_tropical_longitude_deg":
            sun_tropical_longitude_full_deg,

        "moon_tropical_longitude_deg":
            moon_tropical_longitude_full_deg,

        # ----------------------------------------------------
        # Canonical sidereal astronomical quantities
        # ----------------------------------------------------
        "sun_sidereal_longitude_deg":
            sun_sidereal_longitude_full_deg,

        "moon_sidereal_longitude_deg":
            moon_sidereal_longitude_full_deg,

        # ----------------------------------------------------
        # Tithi / Paksha
        # ----------------------------------------------------
        "moon_sun_phase_deg":
            df_tithi[
                "moon_sun_phase_deg"
            ],

        "tithi_number":
            df_tithi[
                "tithi_number"
            ],

        "paksha":
            df_tithi[
                "paksha"
            ],

        "tithi_in_paksha":
            df_tithi[
                "tithi_in_paksha"
            ],

        "tithi_name":
            df_tithi[
                "tithi_name"
            ],

        "tithi_position_deg":
            df_tithi[
                "tithi_position_deg"
            ],

        "tithi_progress":
            df_tithi[
                "tithi_progress"
            ],

        # ----------------------------------------------------
        # Karana
        # ----------------------------------------------------
        "karana_slot":
            df_karana[
                "karana_slot"
            ],

        "karana_name":
            df_karana[
                "karana_name"
            ],

        "karana_half_in_tithi":
            df_karana[
                "karana_half_in_tithi"
            ],

        "karana_position_deg":
            df_karana[
                "karana_position_deg"
            ],

        "karana_progress":
            df_karana[
                "karana_progress"
            ],

        # ----------------------------------------------------
        # Nakshatra
        # ----------------------------------------------------
        "nakshatra_number":
            df_nakshatra[
                "nakshatra_number"
            ],

        "nakshatra_name":
            df_nakshatra[
                "nakshatra_name"
            ],

        "nakshatra_position_deg":
            df_nakshatra[
                "nakshatra_position_deg"
            ],

        "nakshatra_progress":
            df_nakshatra[
                "nakshatra_progress"
            ],

        # ----------------------------------------------------
        # Yoga
        # ----------------------------------------------------
        "yoga_angle_deg":
            df_yoga[
                "yoga_angle_deg"
            ],

        "yoga_number":
            df_yoga[
                "yoga_number"
            ],

        "yoga_name":
            df_yoga[
                "yoga_name"
            ],

        "yoga_position_deg":
            df_yoga[
                "yoga_position_deg"
            ],

        "yoga_progress":
            df_yoga[
                "yoga_progress"
            ],
    },
    index=df_raw.index,
)


# ============================================================
# 3. BUILD CANONICAL RAW F4 MODEL-FEATURE TABLE
# ============================================================
#
# ONLY structured Panchang states are included here.
#
# Deliberately excluded:
#
#     moon_sun_phase_deg
#     sidereal longitudes
#     ayanamsha
#     tithi_progress
#     karana_progress
#     nakshatra_progress
#     yoga_angle_deg
#     yoga_progress
#
# These continuous quantities either overlap with F3 or exist
# only for audit / validation.
#
# tithi is treated as a CATEGORY rather than as an ordinary
# ordinal numerical input.

df_panchang_f4_raw = pd.DataFrame(
    {
        "tithi":
            df_tithi[
                "tithi_number"
            ],

        "paksha":
            df_tithi[
                "paksha"
            ],

        "karana":
            df_karana[
                "karana_name"
            ],

        "nakshatra":
            df_nakshatra[
                "nakshatra_name"
            ],

        "yoga":
            df_yoga[
                "yoga_name"
            ],
    },
    index=df_raw.index,
)


# ============================================================
# 4. MARK ALL FIVE F4 VARIABLES AS CATEGORICAL
# ============================================================
#
# No one-hot, ordinal, sine/cosine, target, or embedding
# encoding is performed here.
#
# Encoding belongs later inside the forecasting pipeline.

for column in df_panchang_f4_raw.columns:

    df_panchang_f4_raw[
        column
    ] = (
        df_panchang_f4_raw[
            column
        ]
        .astype(
            "category"
        )
    )


# ============================================================
# 5. VALIDATE AUDIT TABLE
# ============================================================

print(
    "\nCanonical Panchang audit-table shape:"
)

print(
    df_panchang_audit.shape
)


print(
    "\nCanonical Panchang audit-table "
    "total missing values:"
)

print(
    df_panchang_audit
    .isna()
    .sum()
    .sum()
)


# ============================================================
# 6. VALIDATE RAW F4 TABLE
# ============================================================

print(
    "\nCanonical raw F4 model-feature shape:"
)

print(
    df_panchang_f4_raw.shape
)


print(
    "\nCanonical raw F4 missing values:"
)

print(
    df_panchang_f4_raw
    .isna()
    .sum()
)


# ============================================================
# 7. VERIFY EXPECTED CATEGORY CARDINALITIES
# ============================================================

f4_cardinality = (
    df_panchang_f4_raw
    .nunique()
)


print(
    "\nCanonical F4 category cardinalities:"
)

print(
    f4_cardinality
)


# ============================================================
# 8. VERIFY DATA TYPES
# ============================================================

print(
    "\nCanonical F4 data types:"
)

print(
    df_panchang_f4_raw.dtypes
)


# ============================================================
# 9. CONFIRM CONTINUOUS AUDIT VARIABLES ARE ABSENT FROM F4
# ============================================================

continuous_audit_columns = {
    "lahiri_ayanamsha_extended_deg",
    "moon_sun_phase_deg",
    "tithi_progress",
    "karana_progress",
    "moon_sidereal_longitude_deg",
    "sun_sidereal_longitude_deg",
    "nakshatra_progress",
    "yoga_angle_deg",
    "yoga_progress",
}


continuous_overlap_with_f4 = (
    continuous_audit_columns
    &
    set(
        df_panchang_f4_raw.columns
    )
)


print(
    "\nContinuous audit quantities accidentally "
    "present in F4:"
)

print(
    sorted(
        continuous_overlap_with_f4
    )
)


# ============================================================
# 10. VERIFY CANONICAL F4 INDEX
# ============================================================

print(
    "\nCanonical F4 index equals SOLETE index:"
)

print(
    df_panchang_f4_raw.index.equals(
        df_raw.index
    )
)


# ============================================================
# 11. DISPLAY SAMPLE
# ============================================================

print(
    "\nSample canonical raw F4 Panchang features:"
)

df_panchang_f4_raw.head(12)

Index alignment checks:
{'Tithi': True, 'Karana': True, 'Nakshatra': True, 'Yoga': True}

Canonical Panchang audit-table shape:
(131617, 26)

Canonical Panchang audit-table total missing values:
0

Canonical raw F4 model-feature shape:
(131617, 5)

Canonical raw F4 missing values:
tithi        0
paksha       0
karana       0
nakshatra    0
yoga         0
dtype: int64

Canonical F4 category cardinalities:
tithi        30
paksha        2
karana       11
nakshatra    27
yoga         27
dtype: int64

Canonical F4 data types:
tithi        category
paksha       category
karana       category
nakshatra    category
yoga         category
dtype: object

Continuous audit quantities accidentally present in F4:
[]

Canonical F4 index equals SOLETE index:
True

Sample canonical raw F4 Panchang features:


,tithi,paksha,karana,nakshatra,yoga
Timestamp,,,,,
2018-06-01 00:00:00+00:00,18,Krishna,Vanija,Mula,Shubha
2018-06-01 00:05:00+00:00,18,Krishna,Vanija,Mula,Shubha
2018-06-01 00:10:00+00:00,18,Krishna,Vanija,Mula,Shubha
2018-06-01 00:15:00+00:00,18,Krishna,Vanija,Mula,Shubha
2018-06-01 00:20:00+00:00,18,Krishna,Vanija,Mula,Shubha
2018-06-01 00:25:00+00:00,18,Krishna,Vanija,Purva Ashadha,Shubha
2018-06-01 00:30:00+00:00,18,Krishna,Vanija,Purva Ashadha,Shubha
2018-06-01 00:35:00+00:00,18,Krishna,Vanija,Purva Ashadha,Shubha
2018-06-01 00:40:00+00:00,18,Krishna,Vanija,Purva Ashadha,Shubha


## Independent Cross-Engine Validation of the Final Sidereal Convention

The final F4 implementation uses:

`Skyfield apparent tropical longitude - extended Lahiri ayanamsha`

where the extended Lahiri ayanamsha is obtained from Swiss Ephemeris
`SIDM_LAHIRI` using `get_ayanamsa_ex_ut()`.

Before closing the Panchang feature-engineering notebook, this final
sidereal construction is independently compared against direct
sidereal planetary calculations from the Swiss Ephemeris library.

This validation distinguishes three effects:

1. **Astronomical-engine difference**

   Skyfield tropical solar and lunar longitudes are compared with
   tropical longitudes returned by the Swiss Ephemeris library.

2. **Canonical sidereal agreement**

   The project's final Skyfield + extended-Lahiri sidereal longitudes
   are compared with direct Swiss sidereal calculations.

3. **Legacy/simple ayanamsha sensitivity**

   The earlier `get_ayanamsa_ut()` construction is retained only as a
   diagnostic comparison, demonstrating why the extended convention was
   selected.

The returned Swiss calculation flags are retained explicitly because
they document the computational path actually used by the library.

This section is methodological validation only. It does not modify the
canonical F4 feature tables.

In [16]:
# ============================================================
# CELL 16 — FINAL CROSS-ENGINE SIDEREAL VALIDATION
# ============================================================

# ------------------------------------------------------------
# PURPOSE
# ------------------------------------------------------------
#
# The canonical F4 sidereal convention is already fixed:
#
#     Skyfield apparent tropical longitude
#         -
#     extended Lahiri ayanamsha
#
# This cell independently validates that construction against
# direct sidereal calculations from the Swiss Ephemeris
# library.
#
# We also retain the earlier SIMPLE ayanamsha construction as
# a diagnostic benchmark only.
#
# Nothing in df_panchang_f4_raw is modified here.


# ============================================================
# 1. ENSURE STANDARD LAHIRI MODE
# ============================================================

swe.set_sid_mode(
    swe.SIDM_LAHIRI
)


# ============================================================
# 2. DEFINE SWISS CALCULATION FLAGS
# ============================================================
#
# We request the Swiss ephemeris calculation mode.
#
# IMPORTANT:
#
# Swiss Ephemeris returns flags describing the calculation
# path actually used. These returned flags are retained and
# inspected below rather than assuming the requested engine
# was necessarily available.

SWE_TROPICAL_FLAGS = (
    swe.FLG_SWIEPH
)


SWE_SIDEREAL_FLAGS = (
    swe.FLG_SWIEPH
    |
    swe.FLG_SIDEREAL
)


# ============================================================
# 3. STORAGE ARRAYS
# ============================================================

swe_sun_tropical_test = []
swe_moon_tropical_test = []

swe_sun_sidereal_direct_test = []
swe_moon_sidereal_direct_test = []

swe_tropical_return_flags = []
swe_sidereal_return_flags = []


# ============================================================
# 4. CALCULATE SWISS POSITIONS FOR SIX TEST TIMESTAMPS
# ============================================================

for jd_ut in ayanamsha_test_jd:

    # --------------------------------------------------------
    # Tropical Sun
    # --------------------------------------------------------

    sun_tropical_result, sun_tropical_retflag = (
        swe.calc_ut(
            jd_ut,
            swe.SUN,
            SWE_TROPICAL_FLAGS,
        )
    )


    # --------------------------------------------------------
    # Tropical Moon
    # --------------------------------------------------------

    moon_tropical_result, moon_tropical_retflag = (
        swe.calc_ut(
            jd_ut,
            swe.MOON,
            SWE_TROPICAL_FLAGS,
        )
    )


    # --------------------------------------------------------
    # Direct Lahiri sidereal Sun
    # --------------------------------------------------------

    sun_sidereal_result, sun_sidereal_retflag = (
        swe.calc_ut(
            jd_ut,
            swe.SUN,
            SWE_SIDEREAL_FLAGS,
        )
    )


    # --------------------------------------------------------
    # Direct Lahiri sidereal Moon
    # --------------------------------------------------------

    moon_sidereal_result, moon_sidereal_retflag = (
        swe.calc_ut(
            jd_ut,
            swe.MOON,
            SWE_SIDEREAL_FLAGS,
        )
    )


    # --------------------------------------------------------
    # Longitude is element zero of the returned coordinate
    # tuple.
    # --------------------------------------------------------

    swe_sun_tropical_test.append(
        sun_tropical_result[0]
    )

    swe_moon_tropical_test.append(
        moon_tropical_result[0]
    )

    swe_sun_sidereal_direct_test.append(
        sun_sidereal_result[0]
    )

    swe_moon_sidereal_direct_test.append(
        moon_sidereal_result[0]
    )


    # --------------------------------------------------------
    # Retain actual return flags for reproducibility.
    # --------------------------------------------------------

    swe_tropical_return_flags.append(
        (
            sun_tropical_retflag,
            moon_tropical_retflag,
        )
    )

    swe_sidereal_return_flags.append(
        (
            sun_sidereal_retflag,
            moon_sidereal_retflag,
        )
    )


# ============================================================
# 5. CONVERT TO NUMPY ARRAYS
# ============================================================

swe_sun_tropical_test = np.array(
    swe_sun_tropical_test,
    dtype=float,
)


swe_moon_tropical_test = np.array(
    swe_moon_tropical_test,
    dtype=float,
)


swe_sun_sidereal_direct_test = np.array(
    swe_sun_sidereal_direct_test,
    dtype=float,
)


swe_moon_sidereal_direct_test = np.array(
    swe_moon_sidereal_direct_test,
    dtype=float,
)


# ============================================================
# 6. HELPER — SMALLEST CIRCULAR ANGULAR DIFFERENCE
# ============================================================

def circular_difference_deg(
    angle_a,
    angle_b,
):
    """
    Return the absolute smallest angular separation in degrees.

    This correctly handles the 0° / 360° wrap boundary.
    """

    return np.abs(
        (
            np.asarray(angle_a)
            -
            np.asarray(angle_b)
            +
            180.0
        )
        % 360.0
        -
        180.0
    )


# ============================================================
# 7. COMPARE SKYFIELD AND SWISS TROPICAL GEOMETRY
# ============================================================
#
# These differences measure the underlying astronomical-engine
# / reference-convention disagreement BEFORE any sidereal
# transformation is applied.

sun_tropical_cross_engine_error_deg = (
    circular_difference_deg(
        sun_tropical_longitude_deg,
        swe_sun_tropical_test,
    )
)


moon_tropical_cross_engine_error_deg = (
    circular_difference_deg(
        moon_tropical_longitude_deg,
        swe_moon_tropical_test,
    )
)


print(
    "Maximum Skyfield-vs-Swiss tropical "
    "Sun difference [deg]:"
)

print(
    sun_tropical_cross_engine_error_deg.max()
)


print(
    "\nMaximum Skyfield-vs-Swiss tropical "
    "Moon difference [deg]:"
)

print(
    moon_tropical_cross_engine_error_deg.max()
)


# ============================================================
# 8. VALIDATE CANONICAL EXTENDED-LAHIRI SIDEREAL POSITIONS
# ============================================================
#
# These variables now contain the FINAL canonical positions
# from Cell 4:
#
#     sun_sidereal_longitude_deg
#     moon_sidereal_longitude_deg
#
# Compare them directly against Swiss sidereal positions.

sun_canonical_vs_direct_error_deg = (
    circular_difference_deg(
        sun_sidereal_longitude_deg,
        swe_sun_sidereal_direct_test,
    )
)


moon_canonical_vs_direct_error_deg = (
    circular_difference_deg(
        moon_sidereal_longitude_deg,
        swe_moon_sidereal_direct_test,
    )
)


print(
    "\nMaximum CANONICAL-vs-direct-Swiss "
    "sidereal Sun difference [deg]:"
)

print(
    sun_canonical_vs_direct_error_deg.max()
)


print(
    "\nMaximum CANONICAL-vs-direct-Swiss "
    "sidereal Moon difference [deg]:"
)

print(
    moon_canonical_vs_direct_error_deg.max()
)


# ============================================================
# 9. RECONSTRUCT LEGACY SIMPLE-AYANAMSHA SIDEREAL POSITIONS
# ============================================================
#
# This reconstruction exists ONLY for methodological
# comparison.
#
# It is NOT used anywhere in the final F4 table.

sun_sidereal_simple_legacy_deg = (
    (
        sun_tropical_longitude_deg
        -
        lahiri_ayanamsha_simple_test_deg
    )
    % 360.0
)


moon_sidereal_simple_legacy_deg = (
    (
        moon_tropical_longitude_deg
        -
        lahiri_ayanamsha_simple_test_deg
    )
    % 360.0
)


sun_simple_vs_direct_error_deg = (
    circular_difference_deg(
        sun_sidereal_simple_legacy_deg,
        swe_sun_sidereal_direct_test,
    )
)


moon_simple_vs_direct_error_deg = (
    circular_difference_deg(
        moon_sidereal_simple_legacy_deg,
        swe_moon_sidereal_direct_test,
    )
)


print(
    "\nMaximum LEGACY-simple-vs-direct-Swiss "
    "sidereal Sun difference [deg]:"
)

print(
    sun_simple_vs_direct_error_deg.max()
)


print(
    "\nMaximum LEGACY-simple-vs-direct-Swiss "
    "sidereal Moon difference [deg]:"
)

print(
    moon_simple_vs_direct_error_deg.max()
)


# ============================================================
# 10. QUANTIFY IMPROVEMENT FROM THE CANONICAL CONVENTION
# ============================================================

sun_error_reduction_factor = (
    sun_simple_vs_direct_error_deg.max()
    /
    sun_canonical_vs_direct_error_deg.max()
)


moon_error_reduction_factor = (
    moon_simple_vs_direct_error_deg.max()
    /
    moon_canonical_vs_direct_error_deg.max()
)


print(
    "\nSun maximum-error reduction factor "
    "(simple / canonical):"
)

print(
    sun_error_reduction_factor
)


print(
    "\nMoon maximum-error reduction factor "
    "(simple / canonical):"
)

print(
    moon_error_reduction_factor
)


# ============================================================
# 11. INSPECT ACTUAL SWISS RETURN FLAGS
# ============================================================

unique_tropical_return_flags = sorted(
    set(
        swe_tropical_return_flags
    )
)


unique_sidereal_return_flags = sorted(
    set(
        swe_sidereal_return_flags
    )
)


print(
    "\nUnique tropical calculation return-flag pairs:"
)

print(
    unique_tropical_return_flags
)


print(
    "\nUnique sidereal calculation return-flag pairs:"
)

print(
    unique_sidereal_return_flags
)


# ------------------------------------------------------------
# Diagnostic decoding of relevant flag bits.
#
# This documents whether the library actually used its
# Moshier fallback and whether NONUT appears in the returned
# sidereal flag.
# ------------------------------------------------------------

tropical_uses_moshier = all(
    (
        (sun_flag & swe.FLG_MOSEPH)
        and
        (moon_flag & swe.FLG_MOSEPH)
    )
    for sun_flag, moon_flag
    in swe_tropical_return_flags
)


sidereal_uses_moshier = all(
    (
        (sun_flag & swe.FLG_MOSEPH)
        and
        (moon_flag & swe.FLG_MOSEPH)
    )
    for sun_flag, moon_flag
    in swe_sidereal_return_flags
)


sidereal_return_has_nonut = all(
    (
        (sun_flag & swe.FLG_NONUT)
        and
        (moon_flag & swe.FLG_NONUT)
    )
    for sun_flag, moon_flag
    in swe_sidereal_return_flags
)


print(
    "\nTropical calculations used Moshier fallback:"
)

print(
    bool(
        tropical_uses_moshier
    )
)


print(
    "\nSidereal calculations used Moshier fallback:"
)

print(
    bool(
        sidereal_uses_moshier
    )
)


print(
    "\nReturned sidereal flags contain FLG_NONUT:"
)

print(
    bool(
        sidereal_return_has_nonut
    )
)


# ============================================================
# 12. BUILD SIX-ROW VALIDATION TABLE
# ============================================================

sidereal_cross_engine_results = pd.DataFrame(
    {
        "UTC_timestamp":
            ayanamsha_test_times,

        "lahiri_simple_deg":
            lahiri_ayanamsha_simple_test_deg,

        "lahiri_extended_canonical_deg":
            lahiri_ayanamsha_test_deg,

        "skyfield_sun_tropical_deg":
            sun_tropical_longitude_deg,

        "swiss_sun_tropical_deg":
            swe_sun_tropical_test,

        "canonical_sun_sidereal_deg":
            sun_sidereal_longitude_deg,

        "swiss_sun_sidereal_direct_deg":
            swe_sun_sidereal_direct_test,

        "skyfield_moon_tropical_deg":
            moon_tropical_longitude_deg,

        "swiss_moon_tropical_deg":
            swe_moon_tropical_test,

        "canonical_moon_sidereal_deg":
            moon_sidereal_longitude_deg,

        "swiss_moon_sidereal_direct_deg":
            swe_moon_sidereal_direct_test,
    }
)


# ============================================================
# 13. DISPLAY FINAL VALIDATION TABLE
# ============================================================

print(
    "\nFinal cross-engine sidereal validation:"
)

sidereal_cross_engine_results

Maximum Skyfield-vs-Swiss tropical Sun difference [deg]:
1.6375678654867443e-05

Maximum Skyfield-vs-Swiss tropical Moon difference [deg]:
0.00045761111246633845

Maximum CANONICAL-vs-direct-Swiss sidereal Sun difference [deg]:
1.6375678654867443e-05

Maximum CANONICAL-vs-direct-Swiss sidereal Moon difference [deg]:
0.00045761111246633845

Maximum LEGACY-simple-vs-direct-Swiss sidereal Sun difference [deg]:
0.004809893551112054

Maximum LEGACY-simple-vs-direct-Swiss sidereal Moon difference [deg]:
0.004692779381230139

Sun maximum-error reduction factor (simple / canonical):
293.7217841461722

Moon maximum-error reduction factor (simple / canonical):
10.25495066310335

Unique tropical calculation return-flag pairs:
[(4, 4)]

Unique sidereal calculation return-flag pairs:
[(65604, 65604)]

Tropical calculations used Moshier fallback:
True

Sidereal calculations used Moshier fallback:
True

Returned sidereal flags contain FLG_NONUT:
True

Final cross-engine sidereal validation:


,UTC_timestamp,lahiri_simple_deg,lahiri_extended_canonical_deg,skyfield_sun_tropical_deg,swiss_sun_tropical_deg,canonical_sun_sidereal_deg,swiss_sun_sidereal_direct_deg,skyfield_moon_tropical_deg,swiss_moon_tropical_deg,canonical_moon_sidereal_deg,swiss_moon_sidereal_direct_deg
0,2018-06-01 00:02:00+00:00,24.114318,24.110343,70.472685,70.472701,46.362341,46.362358,277.274831,277.274822,253.164488,253.164479
1,2018-09-01 12:02:00+00:00,24.117856,24.113967,159.004618,159.004616,134.890651,134.890649,48.839032,48.839490,24.725065,24.725523
2,2018-12-01 00:02:00+00:00,24.121317,24.116863,248.725946,248.725960,224.609083,224.609097,171.499213,171.499213,147.382350,147.382350
3,2019-03-01 12:02:00+00:00,24.124778,24.120681,340.605945,340.605944,316.485264,316.485263,284.655756,284.655780,260.535075,260.535099
4,2019-06-01 00:02:00+00:00,24.128278,24.123458,70.245502,70.245491,46.122044,46.122033,40.332366,40.332239,16.208908,16.208781
5,2019-09-01 00:02:00+00:00,24.131797,24.127372,158.288139,158.288142,134.160767,134.160770,180.567801,180.567884,156.440429,156.440512


## Sensitivity to the Legacy Simple Ayanamsha

The final F4 implementation uses the canonical extended Lahiri
ayanamsha.

For methodological transparency, the earlier simple
`get_ayanamsa_ut()` convention is reconstructed across the complete
SOLETE timeline and compared with the final categorical assignments.

This sensitivity analysis does not modify the canonical feature tables.

Its purpose is to quantify how often the approximately 14–18 arcsecond
difference between the two ayanamsha conventions changes a discrete
Panchang state.

Only the ayanamsha-dependent quantities are examined:

- Nakshatra;
- Yoga.

Tithi, Paksha and Karana are excluded because their Moon-Sun angular
difference is invariant to a common ayanamsha.

In [17]:
# ============================================================
# CELL 17 — LEGACY SIMPLE VS CANONICAL EXTENDED SENSITIVITY
# ============================================================

# ------------------------------------------------------------
# PURPOSE
# ------------------------------------------------------------
#
# The canonical F4 table is already complete.
#
# This cell reconstructs the OLD simple-ayanamsha convention
# only to document its effect on:
#
#     Nakshatra
#     Yoga
#
# Nothing in df_panchang_f4_raw is changed.


# ============================================================
# 1. CALCULATE LEGACY SIMPLE AYANAMSHA FOR FULL TIMELINE
# ============================================================

lahiri_ayanamsha_simple_full_deg = np.array(
    [
        swe.get_ayanamsa_ut(
            jd_ut
        )
        for jd_ut in lahiri_jd_full
    ],
    dtype=float,
)


# ============================================================
# 2. COMPARE SIMPLE AND CANONICAL AYANAMSHA
# ============================================================

legacy_minus_canonical_deg = (
    lahiri_ayanamsha_simple_full_deg
    -
    lahiri_ayanamsha_full_deg
)


print(
    "Legacy simple minus canonical extended "
    "ayanamsha range [deg]:"
)

print(
    legacy_minus_canonical_deg.min(),
    "to",
    legacy_minus_canonical_deg.max(),
)


print(
    "\nMaximum absolute ayanamsha difference [deg]:"
)

print(
    np.abs(
        legacy_minus_canonical_deg
    ).max()
)


print(
    "\nMaximum absolute ayanamsha difference [arcsec]:"
)

print(
    np.abs(
        legacy_minus_canonical_deg
    ).max()
    * 3600.0
)


# ============================================================
# 3. RECONSTRUCT LEGACY SIDEREAL LONGITUDES
# ============================================================

moon_sidereal_legacy_deg = (
    (
        moon_tropical_longitude_full_deg
        -
        lahiri_ayanamsha_simple_full_deg
    )
    % 360.0
)


sun_sidereal_legacy_deg = (
    (
        sun_tropical_longitude_full_deg
        -
        lahiri_ayanamsha_simple_full_deg
    )
    % 360.0
)


# ============================================================
# 4. RECONSTRUCT LEGACY NAKSHATRA
# ============================================================

nakshatra_number_legacy = (
    np.floor(
        moon_sidereal_legacy_deg
        / NAKSHATRA_WIDTH_DEG
    )
    .astype(int)
    + 1
)


nakshatra_legacy_change_mask = (
    nakshatra_number_legacy
    !=
    nakshatra_number_full
)


nakshatra_legacy_change_count = (
    nakshatra_legacy_change_mask.sum()
)


print(
    "\nNakshatra assignments differing under "
    "legacy simple ayanamsha:"
)

print(
    nakshatra_legacy_change_count
)


print(
    "\nNakshatra difference percentage [%]:"
)

print(
    100.0
    *
    nakshatra_legacy_change_count
    /
    len(df_raw)
)


# ============================================================
# 5. RECONSTRUCT LEGACY YOGA
# ============================================================

yoga_angle_legacy_deg = (
    (
        moon_sidereal_legacy_deg
        +
        sun_sidereal_legacy_deg
    )
    % 360.0
)


yoga_number_legacy = (
    np.floor(
        yoga_angle_legacy_deg
        / YOGA_WIDTH_DEG
    )
    .astype(int)
    + 1
)


yoga_legacy_change_mask = (
    yoga_number_legacy
    !=
    yoga_number_full
)


yoga_legacy_change_count = (
    yoga_legacy_change_mask.sum()
)


print(
    "\nYoga assignments differing under "
    "legacy simple ayanamsha:"
)

print(
    yoga_legacy_change_count
)


print(
    "\nYoga difference percentage [%]:"
)

print(
    100.0
    *
    yoga_legacy_change_count
    /
    len(df_raw)
)


# ============================================================
# 6. COUNT ROWS AFFECTED IN EITHER SIDEREAL CATEGORY
# ============================================================

any_legacy_sidereal_change_mask = (
    nakshatra_legacy_change_mask
    |
    yoga_legacy_change_mask
)


any_legacy_sidereal_change_count = (
    any_legacy_sidereal_change_mask.sum()
)


both_legacy_sidereal_change_count = (
    nakshatra_legacy_change_mask
    &
    yoga_legacy_change_mask
).sum()


print(
    "\nRows differing in either Nakshatra or Yoga:"
)

print(
    any_legacy_sidereal_change_count
)


print(
    "\nRows differing in both Nakshatra and Yoga:"
)

print(
    both_legacy_sidereal_change_count
)


# ============================================================
# 7. VERIFY CANONICAL F4 TABLE REMAINS UNCHANGED
# ============================================================

print(
    "\nCanonical F4 table shape remains:"
)

print(
    df_panchang_f4_raw.shape
)


print(
    "\nCanonical F4 total missing values remain:"
)

print(
    df_panchang_f4_raw
    .isna()
    .sum()
    .sum()
)


# ============================================================
# 8. SHOW FIRST NAKSHATRA DIFFERENCES
# ============================================================

nakshatra_sensitivity_examples = pd.DataFrame(
    {
        "legacy_sidereal_moon_deg":
            moon_sidereal_legacy_deg[
                nakshatra_legacy_change_mask
            ],

        "canonical_sidereal_moon_deg":
            moon_sidereal_longitude_full_deg[
                nakshatra_legacy_change_mask
            ],

        "legacy_nakshatra":
            nakshatra_number_legacy[
                nakshatra_legacy_change_mask
            ],

        "canonical_nakshatra":
            nakshatra_number_full[
                nakshatra_legacy_change_mask
            ],
    },
    index=df_raw.index[
        nakshatra_legacy_change_mask
    ],
)


print(
    "\nFirst legacy-vs-canonical Nakshatra differences:"
)

print(
    nakshatra_sensitivity_examples.head(10)
)


# ============================================================
# 9. SHOW FIRST YOGA DIFFERENCES
# ============================================================

yoga_sensitivity_examples = pd.DataFrame(
    {
        "legacy_yoga_angle_deg":
            yoga_angle_legacy_deg[
                yoga_legacy_change_mask
            ],

        "canonical_yoga_angle_deg":
            yoga_angle_full_deg[
                yoga_legacy_change_mask
            ],

        "legacy_yoga":
            yoga_number_legacy[
                yoga_legacy_change_mask
            ],

        "canonical_yoga":
            yoga_number_full[
                yoga_legacy_change_mask
            ],
    },
    index=df_raw.index[
        yoga_legacy_change_mask
    ],
)


print(
    "\nFirst legacy-vs-canonical Yoga differences:"
)

print(
    yoga_sensitivity_examples.head(10)
)

Legacy simple minus canonical extended ayanamsha range [deg]:
0.0035533537366880807 to 0.0049057220369057575

Maximum absolute ayanamsha difference [deg]:
0.0049057220369057575

Maximum absolute ayanamsha difference [arcsec]:
17.660599332860727

Nakshatra assignments differing under legacy simple ayanamsha:
43

Nakshatra difference percentage [%]:
0.03267055167645517

Yoga assignments differing under legacy simple ayanamsha:
87

Yoga difference percentage [%]:
0.0661008836244558

Rows differing in either Nakshatra or Yoga:
130

Rows differing in both Nakshatra and Yoga:
0

Canonical F4 table shape remains:
(131617, 5)

Canonical F4 total missing values remain:
0

First legacy-vs-canonical Nakshatra differences:
                           legacy_sidereal_moon_deg  \
Timestamp                                             
2018-06-05 12:25:00+00:00                306.665086   
2018-06-12 13:35:00+00:00                 39.996168   
2018-06-21 19:55:00+00:00                173.332307   
2018

# Notebook 08 — Final Conclusions and Handoff

## Vedic / Panchang Feature Engineering Completed

This notebook constructed, validated and consolidated the initial
Panchang feature family for the SOLETE renewable-energy forecasting
study.

The five selected Panchang quantities are:

1. **Tithi**
2. **Paksha**
3. **Karana**
4. **Nakshatra**
5. **Yoga**

All quantities were calculated for the complete SOLETE 5-minute
timeline of:

`131,617 observations`

with no missing Panchang values.

---

## Astronomical Reference Convention

The astronomical calculations use the same dataset-specific time
interpretation established in Notebook 07:

`astronomical reference time = SOLETE timestamp + 2 minutes`

The original SOLETE timestamp remains the DataFrame index so that all
generated features align directly with the forecasting dataset.

The common astronomical geometry used by F3 and F4 is:

- **Skyfield**
- **JPL DE440s**
- apparent geocentric Sun and Moon positions
- tropical ecliptic longitude
- UTC astronomical reference times

This common geometry is intentional because it keeps the underlying
astronomical information consistent between the F3 and F4 feature
families.

---

## Canonical Lahiri Sidereal Convention

Nakshatra and Yoga require sidereal longitude.

The final project convention is:

**Standard Lahiri ayanamsha**

implemented using:

- Swiss Ephemeris `SIDM_LAHIRI`
- `get_ayanamsa_ex_ut()`
- nutation-inclusive extended ayanamsha

The final sidereal transformation is:

`sidereal longitude = Skyfield tropical longitude - extended Lahiri ayanamsha`

wrapped into:

`[0°, 360°)`

The extended Lahiri ayanamsha across the SOLETE study period is
approximately:

`24.1103° to 24.1274°`

---

## Cross-Engine Validation

The final hybrid sidereal construction was independently compared with
direct sidereal calculations from the Swiss Ephemeris library.

The returned Swiss Ephemeris calculation flags indicated that the
planetary validation calculations used the **Moshier fallback** in the
local installation.

Maximum Skyfield-versus-Swiss tropical-longitude differences across the
six representative validation timestamps were approximately:

- Sun: `1.64 × 10^-5°`
- Moon: `4.58 × 10^-4°`

After applying the canonical extended Lahiri transformation, the
remaining sidereal differences were essentially identical to these
underlying tropical cross-engine differences.

By contrast, the earlier simple `get_ayanamsa_ut()` convention produced
maximum sidereal discrepancies of approximately:

- Sun: `0.00481°`
- Moon: `0.00469°`

The extended convention therefore removes the systematic
simple-ayanamsha offset from the final F4 construction.

---

## Ayanamsha Dependence of Panchang Quantities

The five Panchang quantities divide naturally into two groups.

### Ayanamsha-independent

**Tithi**

Derived from:

`(Moon longitude - Sun longitude) mod 360°`

with 30 sectors of `12°`.

**Paksha**

Derived directly from the Tithi / Moon-Sun phase:

- Shukla
- Krishna

**Karana**

Derived from the same Moon-Sun angular separation with 60 half-Tithi
sectors of `6°`.

Because the same ayanamsha would be subtracted from both the Moon and
Sun, it cancels from their longitude difference.

Therefore:

- Tithi
- Paksha
- Karana

are independent of the selected common ayanamsha.

### Ayanamsha-dependent

**Nakshatra**

Derived from the Moon's canonical Lahiri sidereal longitude using 27
equal sectors of:

`13°20′`

**Yoga**

Derived from:

`(sidereal Sun longitude + sidereal Moon longitude) mod 360°`

using 27 equal sectors of:

`13°20′`

For Yoga, the common ayanamsha enters twice rather than cancelling.

---

## Boundary Validation

Explicit angular boundary tests were performed for:

- Tithi
- Karana
- Nakshatra
- Yoga

including the `0° / 360°` wrap boundary.

The expected left-inclusive sector behavior was confirmed before
full-timeline generation.

Full-timeline consistency checks also confirmed:

- all 30 Tithis are present;
- all 60 Karana slots are present;
- all 11 traditional Karana names are present;
- all 27 Nakshatras are present;
- all 27 Yogas are present;
- no missing Panchang values occur.

---

## Sensitivity to the Legacy Simple Ayanamsha

For methodological transparency, the earlier simple Lahiri convention
was reconstructed and compared with the final canonical extended
convention.

The maximum simple-versus-extended ayanamsha difference across the full
dataset was approximately:

`0.004906°`

or:

`17.66 arcseconds`

This small angular correction affected only observations lying very
close to Panchang sector boundaries.

Across `131,617` observations:

- **43 Nakshatra assignments** differed (`0.0327%`);
- **87 Yoga assignments** differed (`0.0661%`);
- **130 rows** differed in at least one of these quantities;
- no row differed in both Nakshatra and Yoga simultaneously.

This sensitivity analysis supports retaining the more precise extended
Lahiri convention for the final feature-generation pipeline.

---

## Final Panchang Tables

Two separate representations are retained.

### `df_panchang_audit`

Shape:

`(131617, 26)`

This table contains Panchang states together with continuous
astronomical and intermediate quantities required for:

- validation;
- reproducibility;
- diagnostics;
- interpretation.

It is **not** the model-input table.

### `df_panchang_f4_raw`

Shape:

`(131617, 5)`

Columns:

- `tithi`
- `paksha`
- `karana`
- `nakshatra`
- `yoga`

Category cardinalities:

| Feature | Categories |
|---|---:|
| Tithi | 30 |
| Paksha | 2 |
| Karana | 11 |
| Nakshatra | 27 |
| Yoga | 27 |

All five columns are stored as categorical variables.

No missing values are present.

---

## Controlled F3 → F4 Experimental Design

The distinction between F3 and F4 is deliberately preserved.

### F3 — Continuous astronomical representation

F3 contains continuous solar, lunar and Sun-Moon astronomical
quantities.

### F4 — Structured Panchang representation

F4 adds discrete traditional Panchang states:

- Tithi
- Paksha
- Karana
- Nakshatra
- Yoga

Continuous quantities such as:

- Moon-Sun phase angle;
- Lahiri ayanamsha;
- sidereal longitude;
- Tithi progress;
- Karana progress;
- Nakshatra progress;
- Yoga angle;
- Yoga progress

are retained only in the audit table and are **not automatically added
to the F4 model feature table**.

This prevents the F4 experiment from simply reintroducing continuous
astronomical information already present in F3.

---

## Encoding Decision

No numerical encoding of the five F4 categorical variables is performed
in this notebook.

In particular, the notebook does not yet apply:

- ordinal integer encoding;
- one-hot encoding;
- sine/cosine encoding;
- target encoding;
- embeddings.

Encoding will be selected later inside the modelling pipeline so that it
can be fitted using training data only and evaluated consistently under
chronological validation.

---

## Scope Decision

**Masa is intentionally postponed.**

Introducing a lunar month requires additional methodological choices,
including:

- Amanta versus Purnimanta convention;
- lunar-month boundary definition;
- month naming;
- solar ingress relationships;
- handling of exceptional month cases.

Those choices are not necessary for the initial controlled F4
experiment and should not be mixed into the already validated
five-feature Panchang baseline.

---

## Notebook 08 Outcome

The initial F4 Panchang feature-engineering stage is complete.

The final reusable feature family is:

`Tithi + Paksha + Karana + Nakshatra + Yoga`

with:

- validated astronomical timing;
- explicit standard Lahiri convention;
- extended nutation-aware sidereal transformation;
- boundary validation;
- cross-engine validation;
- full-timeline completeness checks;
- clear separation between audit quantities and model inputs.

The next project stage should move from exploratory notebook logic toward
a reusable feature-generation implementation and then integrate F4 into
the controlled chronological forecasting experiments.